# Лабораторная работа 4. Рекомендательные системы, категориальные признаки и адаптивные методы обучения

Результат лабораторной работы − отчет. Мы предпочитаем принимать отчеты в формате ноутбуков IPython (ipynb-файл). Постарайтесь сделать ваш отчет интересным рассказом, последовательно отвечающим на вопросы из заданий. Помимо ответов на вопросы, в отчете также должен быть код, однако чем меньше кода, тем лучше всем: нам − меньше проверять, вам — проще найти ошибку или дополнить эксперимент. При проверке оценивается четкость ответов на вопросы, аккуратность отчета и кода.

## Рекомендательные системы и категориальные признаки

В этой части лабораторной работы мы рассмотрим и сравним несколько различных стратегий для решения задачи рекоммендаций:
- Most popular
- Memory based
- Matrix factorization
- Categorical features based approach (без задания из-за проблем с установкой библиотеками)
- Statistics features based approach

А также научимся замешивать результаты разных стратегий в ограниченный топ с использованием Mixigen алгоритма.

Вам нужно будет провести много экспериментов, поэтому приготовьтесь.

![](https://i.imgur.com/jwyaLjP.jpg)

### Про данные

В этой лабораторной работе будет рассмотрена задача предсказания оценки, которую пользователь поставит фильму. Особенность этой задачи в том, что объекты выборки описываются категориальными признаками, принимающими большое число значений (например: идентификатор пользователя, идентификатор фильма, тэги, киноперсоны).

Мы будем работать с датасетом [MovieLens + IMDb/Rotten Tomatoes](http://files.grouplens.org/datasets/hetrec2011/), файл *hetrec2011-movielens-2k-v2.zip* (18M), [описание](http://files.grouplens.org/datasets/hetrec2011/hetrec2011-movielens-readme.txt). Набор содержит данные о предпочтениях пользователей сервиса рекомендации кинофильмов [MovieLens](http://www.movielens.org/). Пользовательские оценки для фильмов принимают целые значения в интервале от 1 до 5, они записаны в файле *user_ratedmovies.dat* (а так же в *user_ratedmovies-timestamps.dat*,  где для каждой оценки дата и время записаны в формате timestamp), остальные файлы содержат дополнительную информацию о фильмах, которую можно использовать в качестве признаков.
Заметьте: кроме оценок (и тегов), которые пользователь поставил фильмам, про пользователя ничего не известно.

На основании этих данных необходимо построить модель, предсказывающую оценку пользователя фильму, который он еще не смотрел.


- Загрузите данные и создайте разреженную матрицу оценок, где на пересечении $i$-ой строки и $j$-ого столбца стоит рейтинг, который пользователь под номером $i$ поставил фильму под номером $j$, если рейтинг известен, либо ноль, если неизвестен.
- Разбейте данные на обучающую и тестовую выборки так, чтобы в обучающую выборку вошли хронологически первые 70% всех оценок пользователей, а в тестовую хронологически последние 30%.

In [1]:
import os

# os.environ["OPENBLAS_NUM_THREADS"] = "2"
# os.environ["OMP_NUM_THREADS"] = "2"

In [2]:
import pandas as pd
import numpy as np
import torch
import copy
import math

from scipy import sparse
from tqdm.auto import tqdm
from collections import defaultdict


device = "cuda" if torch.accelerator.current_accelerator() else "cpu" 
device

'cpu'

In [4]:
DATA_FILE = "movielens\\user_ratedmovies-timestamps.dat" if os.name == "nt" else  "data/movielens/user_ratedmovies-timestamps.dat"

df = pd.read_csv(DATA_FILE, delimiter="\t")

time_cut = df["timestamp"].quantile(q=0.7)

train_df = df[df["timestamp"] <= time_cut].copy()
test_df = df[df["timestamp"] > time_cut].copy()

train_df = train_df.drop(columns=["timestamp"])
test_df = test_df.drop(columns=["timestamp"])

На подумать...

In [5]:
train_users = train_df["userID"].unique()
train_items = train_df["movieID"].unique()

# Хороший вопрос - а что делать с теми пользователями, который мы не видели в train и с теми фильмами, которых мы не видели
# test_df = test_df[
#     test_df["userID"].isin(train_users) 
#     & test_df["movieID"].isin(train_items)
# ].copy()


In [6]:
len(train_df), len(test_df)

(598918, 256680)

В итоге я убрал из тестовой выборки user'ов, которых не было у нас в train, потому что мы о них ничего не знаем. При этом оставил новые фильмы, потому что по ним можно сделать предсказание на основе общей статистики за все фильмы.

In [135]:
user_ids = np.sort(train_df["userID"].unique())
movie_ids = np.sort(train_df["movieID"].unique())


user_to_idx = {
        user_id: idx
        for idx, user_id in enumerate(user_ids)
}

movie_to_idx = {
    movie_id: idx
    for idx, movie_id in enumerate(movie_ids)
}


rows = train_df["userID"].map(user_to_idx)
cols = train_df["movieID"].map(movie_to_idx)
values = train_df["rating"].astype(float)

train_matrix = sparse.csr_matrix(
    (values, (rows, cols)),
    shape=(len(user_ids), len(movie_ids))
)


train_matrix.shape

(1420, 8895)

### Про оценку качества рекомендаций
Разбиение на train и test необходимо произвести по временной отсечке: фиксируем какое-то время в прошлом: все оценки, выставленные до этого времени, попадают в train, все оценки, выставленные после этого времени, попадают в test.

Тогда все известные рейтинги пользователя $u$ можно представить как $R^u=R^u_{train}\cup R^u_{test}$. Отсутствующие оценки обозначим за $R^u_{unknown}$. 

Выберем некоторого пользователя $u$ и обозначим известные для него рейтинги за $R^u$. 
Для измерения качества рекомендаций в этой лабораторной работе используйте две метрики RMSE и MAP, описанные ниже.

#### RMSE

Метрика [RMSE](https://en.wikipedia.org/wiki/Root-mean-square_deviation) вычисляется следующим образом:
$$ RMSE = \sqrt{ \frac{1}{|users|}\sum_{u\in users}\frac{1}{\left|R^u_{test}\right|} \sum_{i \in R^u_{test}} (r_{ui} - \hat{r}_{ui})^2 },$$
где $r_{ui}$ — наблюдаемая (правильная) оценка, а $\hat{r}_{ui}$ — оценка, предсказанная моделью; $users$ - множество всех пользователей.

Метрика RMSE предназначена для оценки точности предсказания, её удобно оптимизировать напрямую. Данная метрика - хороший выбор для задачи предсказания оценок пользователей в её непосредственной формулировке. Однако следует учесть практический аспект задачи: основная цель рекомендательной системы - рекомендовать фильмы пользователям на основе их оценок, поэтому сами значения оценок на самом деле вторичны. 

#### MAP

Для оценки качества рекомендаций можно использовать метрики качества ранжирования. В этом случае для каждого пользователя $u$ предскажем оценку для всех фильмов из $R^u_{test}$ и $R^u_{unknown}$ и отсортируем эти фильмы по убыванию предсказанного рейтинга. Ожидается, что хороший алгоритм должен выдать релевантные фильмы вверху списка. Обозначим позиции объектов в этом списке за $k^u_i$.

Назовем релевантными те фильмы, которые входят в $R^u_{test}$ и имеют оценку $\ge 3$. Обозначим их за $Rel^u$. Тогда можно вычислить следующую метрику качества рекомендаций для одного пользователя:

$$AP^u=\frac{1}{|Rel^u|} \sum_{i \in Rel^u} P@i,$$
где $$P@i = \frac{1}{i}\sum_{j=1}^i [j\in Rel^u].$$

Усреднив значение этой метрики по всем пользователями, мы получим окончательное значение метрики MAP. Пользователей без релевантных фильмов в тестовой выборке можно не учитывать.

#### Другие способы оценки качества рекомендаций

На практике, как правило, качество рекомендательных систем оценивается в онлайне с помощью [A/B-тестирования](https://en.wikipedia.org/wiki/A/B_testing).

### Most popular

**Задание 1. (1 балл).**
- Постройте рекоммендации на основе **most popular** метода, при котором пользователям рекомендуются объекты в порядке убываниях их популярности (например, среднего рейтинга).
- Оцените качество рекомендаций, вычислив RMSE и MAP.

In [217]:
pred_rating = train_df.groupby("movieID")["rating"].mean()
global_rating_mean = train_df["rating"].mean()  # Предсказание для фильма, который мы не видели в train 

test_df_with_pred = test_df[["userID", "movieID", "rating"]].copy()
test_df_with_pred["pred"] = test_df_with_pred["movieID"].map(lambda x: pred_rating.get(x))

test_df_with_pred.loc[test_df_with_pred["pred"].isna(), "pred"] = global_rating_mean

In [218]:
def compute_rmse(df: pd.DataFrame) -> float:
    temp = pd.concat([df["userID"], (df["rating"] - df["pred"]) ** 2], axis=1)
    return np.sqrt(temp.groupby("userID").mean().iloc[:, 0].mean())


print(f"RMSE: {compute_rmse(test_df_with_pred):.3f}")

RMSE: 0.905


In [14]:
def compute_map_with_unknowns(
    train_df: pd.DataFrame,
    test_df_with_pred: pd.DataFrame,
    relevance_threshold: float = 3.0,
) -> float:

    global_rating_mean = float(train_df["rating"].mean())
    movie_mean = train_df.groupby("movieID")["rating"].mean()

    all_movies = set(movie_mean.index)

    user_train_movies = (
        train_df.groupby("userID")["movieID"]
        .agg(set)
        .to_dict()
    )

    user_test_movies = (
        test_df_with_pred.groupby("userID")["movieID"]
        .agg(set)
        .to_dict()
    )

    user_rel_movies = (
        test_df_with_pred.loc[test_df_with_pred["rating"] >= relevance_threshold]
        .groupby("userID")["movieID"]
        .agg(set)
        .to_dict()
    )

    user_pred_maps = {
        user_id: group.drop_duplicates("movieID").set_index("movieID")["pred"]
        for user_id, group in test_df_with_pred.groupby("userID", sort=False)
    }

    aps = []
    empty_set = set()

    for user_id, rel_movies in user_rel_movies.items():
        if not rel_movies:
            continue

        train_movies = user_train_movies.get(user_id, empty_set)
        test_movies = user_test_movies.get(user_id, empty_set)

        candidate_movies = list((all_movies - train_movies) | test_movies)

        user_pred_map = user_pred_maps[user_id]

        ranking = pd.DataFrame({"movieID": candidate_movies})
        
        ranking["pred"] = ranking["movieID"].map(user_pred_map)
        ranking["pred"] = ranking["pred"].fillna(ranking["movieID"].map(movie_mean))
        ranking["pred"] = ranking["pred"].fillna(global_rating_mean)

        ranking = ranking.sort_values("pred", ascending=False, kind="mergesort")

        hit_count = 0
        precisions = []

        for rank, movie_id in enumerate(ranking["movieID"], start=1):
            if movie_id in rel_movies:
                hit_count += 1
                precisions.append(hit_count / rank)

        ap = sum(precisions) / len(rel_movies)
        aps.append(ap)

    return float(np.mean(aps)) if aps else 0.0


map_score = compute_map_with_unknowns(train_df, test_df_with_pred, 3)
print(f"MAP: {map_score:.3f}")

MAP: 0.027


### Memory based

Теперь рассмотрим [memory-based](https://en.wikipedia.org/wiki/Collaborative_filtering#Memory-based) методы рекоммендаций.
Подход, лежащий в их основе, использует данные о рейтингах для вычисления сходства между пользователями (user-based) или объектами (item-based), на основе этих данных делаются предсказания рейтингов и, в дальнейшем, строятся рекомендации. Эти методы просты в реализации и эффективны на ранних стадиях разработки рекомендательных систем.

Для дальнейшей работы нам понадобится библиотека [Surprise](http://surpriselib.com/). Эта библиотека заточена под оценивание рейтингов (explicit feedback).

**Задание 2. (1 балл).**
- Постройте рекомендации на основе [item-based](https://surprise.readthedocs.io/en/stable/knn_inspired.html) подхода, реализованном в библиотеке Surprise (обратите внимание на параметр *user_based* в словаре *sim_options*).
- Оцените качество рекомендаций в зависимости от выбранной функции похожести *msd/cosine/pearson* по каждой из метрик: RMSE, MAP.

In [ ]:
from surprise.prediction_algorithms import knns
from surprise import Dataset, Reader, Prediction


def build_dataset(df: pd.DataFrame) -> Dataset:
    min_r, max_r = min(df["rating"].min(), test_df["rating"].min()), max(df["rating"].max(), test_df["rating"].max())
    reader = Reader(rating_scale=(min_r, max_r))
    
    return Dataset.load_from_df(
        df[["userID", "movieID", "rating"]],
        reader
    )


def build_df_from_predictions(predictions: Prediction) -> pd.DataFrame:
    return pd.DataFrame([
            {
                "userID": pred.uid,
                "movieID": pred.iid,
                "rating": pred.r_ui,
                "pred": pred.est,
            }
            for pred in predictions
        ])


def build_df_from_matrix(predictions: sparse.coo_matrix) -> pd.DataFrame:
    return pd.DataFrame({
                "userID": predictions.row,
                "movieID": predictions.col,
                "pred": predictions.data,
            })
    

def surprise_fit_predict(algo, train_df: pd.DataFrame, test_df: pd.DataFrame):
    train_data = build_dataset(train_df)
    trainset = train_data.build_full_trainset()
    
    algo.fit(trainset)
    
    testset = list(
        test_df[["userID", "movieID", "rating"]].itertuples(index=False, name=None)
    )
    predictions = algo.test(testset)
    
    return predictions



def surprise_make_expriment(algo, train_df: pd.DataFrame, test_df: pd.DataFrame):
    predictions = surprise_fit_predict(algo, train_df, test_df)
    test_df_with_preds = build_df_from_predictions(predictions)
    
    return {
        "rmse": compute_rmse(test_df_with_preds),
        "map": compute_map_with_unknowns(train_df, test_df_with_preds)
    }

In [10]:
sim_funcs = ["msd", "cosine", "pearson"]

for sim_func in sim_funcs:
    algo = knns.KNNBasic(
        k=40,
        min_k=1,
        sim_options={
            "name": sim_func,
            "user_based": False
        },
        verbose=False      
    )
    
    metrics = surprise_make_expriment(algo, train_df, test_df)
    
    print(f"{sim_func}: {metrics}")

msd: {'rmse': 0.9400415265781729, 'map': 0.02462935054174958}
cosine: {'rmse': 0.9667931108341016, 'map': 0.024971810900598922}
pearson: {'rmse': 0.963505986825794, 'map': 0.021397961175524435}


Лучшее ранжирование показало косинусное расстояние

### Matrix factorization

**Задание 3. (1.5 балла).**
- Разложите матрицу рейтингов с помощью [разреженного SVD](http://scikit-learn.org/stable/modules/generated/sklearn.decomposition.TruncatedSVD.html) и, восстановив ее, получите предсказания рейтингов для всех пар пользователь-объект. В данном случае неизвестные рейтинги заполняются нулями, а затем восстанавливаются с помощью SVD (этот метод называется PureSVD).
- Рассмотрите, как минимум, 5 различных значений ранга разложения. Оцените качество рекомендаций, используя описанные выше метрики: RMSE, MAP. Для обеих метрик постройте графики зависимости качества алгоритма от выбранного ранга разложения.

In [68]:
from sklearn.decomposition import TruncatedSVD

In [17]:
def sklearn_svd_fit_predict(train_matrix: sparse, rank: int):
    svd = TruncatedSVD(
        n_components=rank,
        n_iter=7
    )

    Z = svd.fit_transform(train_matrix)
    return svd.inverse_transform(Z)


def svd_get_df_pred(test_df: pd.DataFrame, pred_matrix: sparse) -> pd.DataFrame:
    test_df_with_pred = test_df.copy()

    # Нужно преобразовать id в индексы матрицы
    test_df_with_pred["u_idx"] = (
        test_df_with_pred["userID"]
        .map(user_to_idx)
    )

    test_df_with_pred["i_idx"] = (
        test_df_with_pred["movieID"]
        .map(movie_to_idx)
    )



    # Нужно убрать все фильмы и юзеры, которых не было в train
    test_df_with_pred = test_df_with_pred.dropna(
        subset=["u_idx", "i_idx"]
    ).copy()

    test_df_with_pred["u_idx"] = (
        test_df_with_pred["u_idx"]
        .astype(int)
    )

    test_df_with_pred["i_idx"] = (
        test_df_with_pred["i_idx"]
        .astype(int)
    )
    
    
    
    
    test_df_with_pred["pred"] = pred_matrix[
            test_df_with_pred["u_idx"].to_numpy(),
            test_df_with_pred["i_idx"].to_numpy()
    ]


    min_rating = train_df["rating"].min()
    max_rating = train_df["rating"].max()

    test_df_with_pred["pred"] = (
        test_df_with_pred["pred"]
        .clip(min_rating, max_rating)
    )
    
    
    return test_df_with_pred


In [ ]:
ranks = [2, 5, 10, 50, 100, 500]

for rank in ranks:
    pred_matrix = sklearn_svd_fit_predict(train_matrix, rank)
    test_df_with_pred = svd_get_df_pred(test_df, pred_matrix)
    
    print(f"Rank: {rank}")
    print(f"RMSE: {compute_rmse(test_df_with_pred)}")
    print(f"MAP: {compute_map_with_unknowns(train_df, test_df_with_pred)}\n")

Rank: 2
RMSE: 2.8733053446969117
MAP: 0.0038810845771402986

Rank: 5
RMSE: 2.821133653569489
MAP: 0.003695309755135704

Rank: 10
RMSE: 2.800404081739604
MAP: 0.0037870935939734794

Rank: 50
RMSE: 2.8035139460470546
MAP: 0.0034149298262880585

Rank: 100
RMSE: 2.8453887279938774
MAP: 0.003264381296563368

Rank: 500
RMSE: 3.0920700674091086
MAP: 0.0031985326771790104



- На основании полученных данных ответьте на вопросы: как значение ранга влияет на метрики и почему именно так?

При изменении ранга, у меня получилось найти более менее оптимальный ранг по матрикам (``rank=10``). Увеличивая или уменьшая ранг от 10, мы замечаем заметное ухудшение качества по всем метрикам. Я думаю это связано с тем, что матрица ``train_matrix`` очень сильно разряженная, то есть ``SVD`` учитывает в своём разложении огромное количество нулей, что делает предсказания слишком шумными.  
Помимо этого, видно, как с ростом ``rank`` начинает быстро расти ``RMSE``, связано это с переобучением ``SVD`` под шум данных.  

**Задание 4. (2 балла).** В библиотеке Surprise также есть  алгоритмы рекомендации, основанный на матричном разложении:  [SVD, SVDpp, NMF](https://surprise.readthedocs.io/en/stable/matrix_factorization.html). Проведите эксперименты из предыдущего задания для алгоритма SVD, рассмотрев, как влияют различные параметры на результат. Сравните результаты экспериментов с полученными ранее результатами.

Мы рекомендуем при проведении экспериментов обратить внимание на следующие параметры:
 - n_factors;
 - biased;
 - набор параметров learning rate (всегда важно для алгоритма градиентого спуска);
 - набор параметров регуляризации.


In [18]:
from surprise.prediction_algorithms.matrix_factorization import SVD

In [15]:
n_factors = [50, 100, 200]

lrs_all = [0.005, 1e-3]
regs_all = [0.02, 0.1]

for n_factor in n_factors:
    for biased in [True, False]:
        for lr_all in lrs_all:
            for reg_all in regs_all:

                svd = SVD(
                    n_factors=n_factor,
                    biased=biased,

                    lr_all=lr_all,
                    reg_all=reg_all
                )

                options = {"n_factor": n_factor, "baised": biased, "lr_all": lr_all, "reg_all": reg_all}
                metrics = surprise_make_expriment(algo, train_df, test_df)
                
                print(f"Expr. {options}\nMetrics: {metrics}\n")


Expr. {'n_factor': 50, 'baised': True, 'lr_all': 0.005, 'reg_all': 0.02}
Metrics: {'rmse': 0.963505986825794, 'map': 0.021397961175524435}

Expr. {'n_factor': 50, 'baised': True, 'lr_all': 0.005, 'reg_all': 0.1}
Metrics: {'rmse': 0.963505986825794, 'map': 0.021397961175524435}

Expr. {'n_factor': 50, 'baised': True, 'lr_all': 0.001, 'reg_all': 0.02}
Metrics: {'rmse': 0.963505986825794, 'map': 0.021397961175524435}

Expr. {'n_factor': 50, 'baised': True, 'lr_all': 0.001, 'reg_all': 0.1}
Metrics: {'rmse': 0.963505986825794, 'map': 0.021397961175524435}

Expr. {'n_factor': 50, 'baised': False, 'lr_all': 0.005, 'reg_all': 0.02}
Metrics: {'rmse': 0.963505986825794, 'map': 0.021397961175524435}

Expr. {'n_factor': 50, 'baised': False, 'lr_all': 0.005, 'reg_all': 0.1}
Metrics: {'rmse': 0.963505986825794, 'map': 0.021397961175524435}

Expr. {'n_factor': 50, 'baised': False, 'lr_all': 0.001, 'reg_all': 0.02}
Metrics: {'rmse': 0.963505986825794, 'map': 0.021397961175524435}

Expr. {'n_factor': 5

#### SLIM

Пока отложем все библиотеки в сторону, пора достать с дальней полки GPU (если еще не), стряхнуть с него пыль и воспользоваться им при решении этого задания.


Более подробно алгоритм SLIM описан в [оригинальной статье](https://www.researchgate.net/publication/220765374_SLIM_Sparse_Linear_Methods_for_Top-N_Recommender_Systems). Оригинально, этот алгоритм используется именно для implicit feedback-а из бинаризации strong сигнала.

Возьмите код по SLIM с семинара и воспользуйтесь им для решения задачи кандидатогенерации/top-n-recommendation (MAP). Учтите что в базовой постановке мы не используем рейтинги, а бинаризуем их, поэтому оценивать RMSE не получится.

Бинаризируем датасет

In [107]:
def binarize_df(df: pd.DataFrame, threshold: float = 3.0) -> pd.DataFrame:
    bin_df = df[df["rating"] >= threshold].copy()
    bin_df["rating"] = 1.0
    return bin_df


def build_binary_matrix(df: pd.DataFrame):

    user_to_idx = {u: i for i, u in enumerate(user_ids)}
    movie_to_idx = {m: i for i, m in enumerate(movie_ids)}
    idx_to_user = {i: u for u, i in user_to_idx.items()}
    idx_to_movie = {i: m for m, i in movie_to_idx.items()}

    rows = df["userID"].map(user_to_idx).to_numpy()
    cols = df["movieID"].map(movie_to_idx).to_numpy()
    data = np.ones(len(df), dtype=np.float32)

    X_csr = sparse.csr_matrix(
        (data, (rows, cols)),
        shape=(len(user_ids), len(movie_ids))
    )
    X_csc = X_csr.tocsc()

    return X_csr, X_csc, user_to_idx, movie_to_idx, idx_to_user, idx_to_movie



train_bin_df = binarize_df(train_df)
(
    X_csr, X_csc,
    user_to_idx, movie_to_idx,
    idx_to_user, idx_to_movie
) = build_binary_matrix(train_bin_df)

print(train_bin_df.head())
print(X_csr.shape)

   userID  movieID  rating      timestamp
1      75       32     1.0  1162160624000
2      75      110     1.0  1162161008000
4      75      163     1.0  1162160970000
5      75      165     1.0  1162160715000
6      75      173     1.0  1162160257000
(1420, 8895)


**Задание 5. (3 балла).**

Чтобы SLIM можно было по человечески пользоваться, нашу реализацию нужно ускорить. Поищите узкие места и ускорьте код.

Несколько инсайдов по тому, где искать:
- перед оптимизацией весов для каждого айтема посчитать нормы один раз и передавать в функцию coordinate-descent
- не пересчитывать матрицу скоров для каждого j-го айтема (`R_csr @ w`), а посчитать ее один раз и инкрементально изменять
- попробуйте разные методы остановки: когда количество ненулевых компонент перестало изменяться, максимальный вес перестал изменяться и т.д.
- SLIM with Feature Selection
- и т.д.


Провалидируйте, что результат ускоренной реализации не сильно отличается от результата наивной реализации (выученные веса для ряда айтемов). Какого ускорения удалось добиться при расчете на всех айтемах?

Чтобы посчитать ускорение на всех айтемах не нужно инферинсить ни медленную ни быструю модель на всех данных, оцените их скорость приблезительно.

После ускорения вам нужно замерить качество работы вашего алгоритма. Для того чтобы это сделать, вам нужно научиться по пользователю рекомендовать товары. Воспользуйтесь для этого способом описанным в `Efficient Top-N Recommendation from SLIM` главе статьи.


Поскольку SLIM работает на основе линейной модели, которая предсказывает оценку item'а пользователем на основе других item'ов, для ускорения можно добавить отбор товаров, по которым мы будем обучаться и строить предсказания, что ускорит вычисления и уберёт ненужный шум матрицы взаимодействий

In [109]:
def precompute_gram(X_csr: sparse.csr_matrix) -> np.ndarray:
    G = (X_csr.T @ X_csr).astype(np.float32).toarray()
    return G


def build_topk_candidates_from_gram(
    G_np: np.ndarray,
    top_k: int = 200,
    min_cooc: float = 1.0,  # Убираем item, которые встретились только у одного пользователя, т.к. это шум
):
    n = G_np.shape[0]
    candidates = []

    for i in tqdm(range(n), desc="Feature selection"):
        row = G_np[i].copy()
        row[i] = 0.0

        if top_k is None or top_k >= n - 1:
            idx = np.flatnonzero(row >= min_cooc)
        else:
            idx = np.argpartition(row, -top_k)[-top_k:]
            idx = idx[row[idx] >= min_cooc]
            idx = idx[np.argsort(row[idx])[::-1]]

        candidates.append(idx.astype(np.int32, copy=False))

    return candidates

Функция обучения ``SLIM FISTA`` алгоритм.

In [110]:
def power_iteration_lipschitz(H: torch.Tensor, n_iter: int = 20) -> float:
    v = torch.randn(H.shape[0], device=H.device, dtype=H.dtype)
    v = v / (v.norm() + 1e-12)

    for _ in range(n_iter):
        v = H @ v
        v = v / (v.norm() + 1e-12)

    Hv = H @ v
    L = torch.dot(v, Hv).abs().clamp_min(1e-12).item()
    return L


def slim_fista_one_item(
    G: torch.Tensor,
    target_idx: int,
    candidate_idx: np.ndarray,
    lambda1: float = 0.01,
    lambda2: float = 1e-6,
    max_iters: int = 50,
    tol: float = 1e-5,
    device=device
):
    """
    Решает задачу для одного target item на reduced candidate set.
    """
    if candidate_idx.size == 0:
        return np.array([], dtype=np.int32), np.array([], dtype=np.float32)

    idx = candidate_idx[candidate_idx != target_idx]
    if idx.size == 0:
        return np.array([], dtype=np.int32), np.array([], dtype=np.float32)

    idx_t = torch.as_tensor(idx, dtype=torch.long, device=device)

    G_sub = G.index_select(0, idx_t).index_select(1, idx_t)
    g = G.index_select(0, idx_t)[:, target_idx]

    H = G_sub + lambda2 * torch.eye(len(idx), device=device, dtype=G.dtype)
    L = power_iteration_lipschitz(H, n_iter=15)
    step = 1.0 / L

    w = torch.zeros(len(idx), device=device, dtype=G.dtype)
    y = w.clone()
    t = 1.0

    with torch.no_grad():
        for _ in range(max_iters):
            grad = G_sub @ y - g + lambda2 * y
            z = y - step * grad
            w_new = torch.clamp(z - lambda1 * step, min=0.0)

            diff = (w_new - w).abs().max().item()
            if diff < tol:
                w = w_new
                break

            t_new = 0.5 * (1.0 + np.sqrt(1.0 + 4.0 * t * t))
            y = w_new + ((t - 1.0) / t_new) * (w_new - w)
            w = w_new
            t = t_new

    w_np = w.detach().cpu().numpy()
    return idx, w_np

In [111]:
def train_slim_fista(
    X_csr: sparse.csr_matrix,
    lambda1: float = 0.01,
    lambda2: float = 1e-6,
    max_iters: int = 30,
    top_k: int = 200,
    min_cooc: float = 1.0,
    device=device
):
    G_np = precompute_gram(X_csr)
    candidates = build_topk_candidates_from_gram(
        G_np,
        top_k=top_k,
        min_cooc=min_cooc,
    )

    G = torch.from_numpy(G_np).to(device)

    n_items = X_csr.shape[1]
    rows, cols, data = [], [], []

    for item_idx in tqdm(range(n_items), desc="Training SLIM-FISTA"):
        idxs, w = slim_fista_one_item(
            G=G,
            target_idx=item_idx,
            candidate_idx=candidates[item_idx],
            lambda1=lambda1,
            lambda2=lambda2,
            max_iters=max_iters,
            tol=1e-5,
            device=device,
        )

        if idxs.size == 0:
            continue

        nz = w > 0
        if np.any(nz):
            rows.extend(idxs[nz].tolist())
            cols.extend([item_idx] * int(nz.sum()))
            data.extend(w[nz].tolist())

    W_csr = sparse.csr_matrix(
        (
            np.array(data, dtype=np.float32),
            (np.array(rows, dtype=np.int32), np.array(cols, dtype=np.int32))
        ),
        shape=(n_items, n_items),
        dtype=np.float32
    )
    W_csr.eliminate_zeros()
    return W_csr

In [112]:
W_csr = train_slim_fista(
    X_csr=X_csr,
    lambda1=0.01,
    lambda2=1e-6,
    max_iters=30,
    top_k=200,
    min_cooc=1.0,
    device=device
)


Feature selection:   0%|          | 0/8895 [00:00<?, ?it/s]

Training SLIM-FISTA:   0%|          | 0/8895 [00:00<?, ?it/s]

In [113]:
W_csr.shape

(8895, 8895)

In [114]:
def slim_predict_scores(X_csr: sparse.csr_matrix, W_csr: sparse.csr_matrix) -> np.ndarray:
    scores = X_csr.dot(W_csr)
    if sparse.issparse(scores):
        scores = scores.toarray()
    return np.asarray(scores, dtype=np.float32)


def compute_map_from_scores(
    train_df: pd.DataFrame,
    test_df: pd.DataFrame,
    scores_matrix: np.ndarray,
    user_to_idx: dict,
    movie_to_idx: dict,
    relevance_threshold: float = 3.0,
) -> float:
    n_items = scores_matrix.shape[1]

    user_train_movies = (
        train_df.groupby("userID")["movieID"]
        .agg(set)
        .to_dict()
    )

    rel_test = test_df[test_df["rating"] >= relevance_threshold]
    rel_movies_by_user = (
        rel_test.groupby("userID")["movieID"]
        .agg(set)
        .to_dict()
    )

    aps = []

    for user_id, rel_movies in rel_movies_by_user.items():
        if user_id not in user_to_idx:
            continue

        rel_idx = np.array(
            [movie_to_idx[m] for m in rel_movies if m in movie_to_idx],
            dtype=np.int32
        )
        if rel_idx.size == 0:
            continue

        u = user_to_idx[user_id]
        scores = scores_matrix[u].copy()

        seen_train = user_train_movies.get(user_id, set())
        seen_idx = [movie_to_idx[m] for m in seen_train if m in movie_to_idx]
        scores[seen_idx] = -np.inf

        ranked_idx = np.argsort(-scores, kind="mergesort")

        rel_mask = np.zeros(n_items, dtype=bool)
        rel_mask[rel_idx] = True

        hits = rel_mask[ranked_idx]
        hit_pos = np.flatnonzero(hits) + 1

        if hit_pos.size == 0:
            aps.append(0.0)
            continue

        cum_hits = np.cumsum(hits)[hit_pos - 1]
        ap = float(np.mean(cum_hits / hit_pos))
        aps.append(ap)

    return float(np.mean(aps)) if aps else 0.0


In [115]:
scores_matrix = slim_predict_scores(X_csr, W_csr)

In [ ]:
map_score = compute_map_from_scores(
    train_df=train_bin_df,  # считаю именно по positive взаимодействиям 
    test_df=test_df,
    scores_matrix=scores_matrix,
    user_to_idx=user_to_idx,
    movie_to_idx=movie_to_idx,
    relevance_threshold=3.0
)

print(f"MAP: {map_score:.4f}")

MAP: 0.0909


Я пытался оптимизировать SLIM с семинара, ничего не получилось, всё равно считалось всё слишком долго, поэтому я использовал FISTA.

**Задание 6. (2 балла).** Хоть у нас таргеты и бинаризированы, но мы можем учитывать рейтинги в развесовке семплов/айтемов. Давайте это поддержим. Есть несколько способов это учесть.

Давайте модифицируем алгоритм и посмотрим на качество рекомендаций:
- В нашей модели мы положили $r_{ui} = R_{ui} \in \{0, 1\}$, тем самым потеряв некотый сигнал о силе рейтинга уверенность в рейтинге. Положите информацию о рейтнге в матрицу, вместо бинарного значения: $r_{ui} = R_{ui} = 1 + \text{rating}_{ui} * c$, где $c=10$
- В нашей модели мы равнозрачно используем рейтинги фильмов, но это может быть плохо из-за того, что ошибки при приближении большого рейтинга более важны чем ошибки при приближении более маленького. Из-за этого предлагается ввести развесовку $c_{ui} = 1 + r_{ui} * 10$.

In [117]:
def build_weighted_matrices(
    df: pd.DataFrame,
    rating_scale: float = 10.0,
    confidence_scale: float = 10.0,
):
    """
    Строит:
        R_csr — target matrix
        C_csr — confidence matrix

    r_ui = 1 + rating * rating_scale
    c_ui = 1 + r_ui * confidence_scale
    """

    user_ids = np.sort(df["userID"].unique())
    movie_ids = np.sort(df["movieID"].unique())

    user_to_idx = {u: i for i, u in enumerate(user_ids)}
    movie_to_idx = {m: i for i, m in enumerate(movie_ids)}

    idx_to_user = {i: u for u, i in user_to_idx.items()}
    idx_to_movie = {i: m for m, i in movie_to_idx.items()}

    rows = df["userID"].map(user_to_idx).to_numpy()
    cols = df["movieID"].map(movie_to_idx).to_numpy()

    ratings = df["rating"].astype(np.float32).to_numpy()

    # r_ui
    r_vals = 1.0 + rating_scale * ratings

    # c_ui
    c_vals = 1.0 + confidence_scale * r_vals

    shape = (len(user_ids), len(movie_ids))

    R_csr = sparse.csr_matrix(
        (r_vals, (rows, cols)),
        shape=shape,
        dtype=np.float32
    )

    C_csr = sparse.csr_matrix(
        (c_vals, (rows, cols)),
        shape=shape,
        dtype=np.float32
    )

    return (
        R_csr,
        C_csr,
        user_to_idx,
        movie_to_idx,
        idx_to_user,
        idx_to_movie
    )

In [118]:
(
    R_csr,
    C_csr,
    user_to_idx,
    movie_to_idx,
    idx_to_user,
    idx_to_movie
) = build_weighted_matrices(train_df)

In [119]:
def soft_threshold(x, lam):
    return torch.sign(x) * torch.clamp(torch.abs(x) - lam, min=0.0)


def slim_fista_one_item_weighted(
    R_csr,
    C_csr,
    target_idx,
    candidate_idx,
    lambda1=0.01,
    lambda2=1e-4,
    max_iters=100,
    tol=1e-5,
    device=device
):
    r_i = R_csr[:, target_idx].toarray().reshape(-1).astype(np.float32)
    c_i = C_csr[:, target_idx].toarray().reshape(-1).astype(np.float32)

    r_i = torch.tensor(r_i, device=device)
    c_i = torch.tensor(c_i, device=device)

    X_sub = R_csr[:, candidate_idx].toarray().astype(np.float32)
    X = torch.tensor(X_sub, device=device)
    n_features = X.shape[1]

    sqrt_c = torch.sqrt(c_i).unsqueeze(1)
    Xw = X * sqrt_c

    svals = torch.linalg.svdvals(Xw)

    L = svals[0] ** 2 + lambda2
    step = 1.0 / L


    w = torch.zeros(n_features, device=device)
    y = w.clone()
    t = 1.0
    for _ in range(max_iters):

        w_old = w.clone()
        pred = X @ y
        residual = c_i * (pred - r_i)

        grad = X.T @ residual + lambda2 * y
        z = y - step * grad

        w = soft_threshold(z, lambda1 * step)
        w = torch.clamp(w, min=0.0)

        t_new = 0.5 * (1 + np.sqrt(1 + 4 * t * t))

        y = w + ((t - 1) / t_new) * (w - w_old)
        t = t_new

        diff = torch.max(torch.abs(w - w_old))

        if diff < tol:
            break

    return w.detach().cpu().numpy()


In [120]:
def train_slim_weighted_fista(
    R_csr,
    C_csr,
    lambda1=0.01,
    lambda2=1e-4,
    top_k=300,
    max_iters=100,
    tol=1e-5,
    device=device
):
    """
    Обучение всей sparse matrix W
    """

    n_items = R_csr.shape[1]
    
    G = precompute_gram(R_csr)
    candidates = build_topk_candidates_from_gram(G, top_k=top_k)

    W = sparse.lil_matrix((n_items, n_items), dtype=np.float32)
    for item_idx in tqdm(range(n_items), desc="SLIM"):
        candidate_idx = candidates[item_idx]

        if len(candidate_idx) == 0:
            continue

        w = slim_fista_one_item_weighted(
            R_csr=R_csr,
            C_csr=C_csr,
            target_idx=item_idx,
            candidate_idx=candidate_idx,
            lambda1=lambda1,
            lambda2=lambda2,
            max_iters=max_iters,
            tol=tol,
            device=device,
        )

        W[candidate_idx, item_idx] = w

    return W.tocsr()

In [121]:
W_csr = train_slim_weighted_fista(
    R_csr=R_csr,
    C_csr=C_csr,
    lambda1=0.01,
    lambda2=1e-4,
    top_k=200,
    max_iters=20, 
    device=device
)

Feature selection:   0%|          | 0/8895 [00:00<?, ?it/s]

SLIM:   0%|          | 0/8895 [00:00<?, ?it/s]

In [122]:
pred_matrix = slim_predict_scores(
    R_csr,
    W_csr
)

In [123]:
map_score = compute_map_from_scores(
    train_df=train_df,
    test_df=test_df,
    scores_matrix=pred_matrix,
    user_to_idx=user_to_idx,
    movie_to_idx=movie_to_idx,
    relevance_threshold=3.0,
)

print(f"MAP weighted: {map_score:.4f}")

MAP weighted: 0.0751


Как можнно заметить, weighted SLIM не дал большого прироста к метрике, возможно это связано с тем, что нужно использовать больше кадидатов или итераций для лучшей сходимости, но это очень долго.  
Либо, ещё это может быть связано с тем, что при рассчёте матрицы Грамма для отбора кандидатов, каждый элемент матрицы $G_{ij}$, в случае $r_{ui} = R_{ui} = 1 + \text{rating}_{ui} * c$, уже не всегда отображает смысл "насколько часто фильм $i$ смотрят вместе с фильмом $j$".

**Задание 7. (2 бонусных балла)** [EASE](https://arxiv.org/abs/1905.03375)

В качестве бонусного задания вам предлагаеся обучить EASE модель для кандидатогенерации. Ее можно считать расширением SLIM модели, с одним важным отличием - мы теперь работаем над dense данными, а не на sparse.

Но при этом мы материализуем не user item матрицу, которая даже для учебных датасетов может не вмещаться в память, а матрицу item-item, из-за чего получаем возможность работать с dense форматом.



In [124]:
def get_ease_solution(X_csc: sparse.csc_matrix, lambda_: float) -> sparse.csr_array:
    G = (X_csc.T @ X_csc).toarray().astype(np.float32)
    
    diag_idx = np.arange(G.shape[0])
    G[diag_idx, diag_idx] += lambda_
    
    P = np.linalg.inv(G)
    W = -P / np.diag(P)
    
    np.fill_diagonal(W, 0.0)
    
    return sparse.csr_array(W)


In [125]:
W_ease = get_ease_solution(X_csc, 1.0)
pred_matrix = slim_predict_scores(X_csr, W_ease)

In [126]:
map_score = compute_map_from_scores(
    train_df=train_df,
    test_df=test_df,
    scores_matrix=pred_matrix,
    user_to_idx=user_to_idx,
    movie_to_idx=movie_to_idx,
    relevance_threshold=3.0,
)

print(f"MAP EASE: {map_score:.4f}")

MAP EASE: 0.0591


### Сategorical features based approach

В этой части задания мы рассмотрим подход к рекомендациям на основе категориальных разреженных признаков. В данном случае это id-пользователя и id-фильма, а также вам будет необходимо добавить один или несколько признаков из имеющихся данных, например: жанр фильма, киноперсоны из фильма, последний оцененный пользователем фильм, средняя оценка пользователя, ...

#### Разреженные признаки

В данной части работы вам необходимо создать разреженную матрицу данных, закодировав каждый из категориальных признаков вектором чисел. Примером может служить следующая иллюстрация добавления различных признаков:

![](http://i.imgur.com/7nUMFx5.png)

Здесь, для наглядности, вся матрица объект-признак разбита на части, каждая из которых соответствует одному или группе категориальных признаков. Например, часть *User* соответствует закодированному признаку *user_id*, часть *Movie* — признаку *movie_id*, *Other movies rated* содержит в себе оценки пользователя другим фильмам, а *Last movie rated* соответствует признаку "последний оцененный пользователем фильм".

Одно из самых популярных библиотек для работы с факторизационными машинами будет [LightFM](https://github.com/lyst/lightfm), которая дает хорошие бейзлайны. Так как есть определеные сложности с установкой всех библиотек для факторизационных машин, то в этом блоке мы вас попросим только разобраться с тем, какой лосс они оптимизируют.

#### LightFM

Библиотека [LightFM](https://github.com/lyst/lightfm) реализует общий подход к задачам с категориальными признаками, в основе которого лежат факторизационные машины.

Для оптимизации она использует ранжирующий WARP лосс. В чем его идея? Разберитесь самостоятельно, это понадобится в дальнейшем.

### Statistics features based approach

В качестве иллюстрации, что такое признаки-статистики или признаки-счётчики, рассмотрим категориальный признак "жанр фильма". Для каждого жанра мы можем посчитать некоторое числовое значение, например, среднее значение оценок фильмов этого жанра. Затем, если в матрице данных мы заменим значения жанров этого категориального признака на соответствующие значения средних рейтингов данных жанров, то получим новый числовой признак-счётчик.

Здесь стоит обратить внимание на следующее:

1. При создании таких счётчиков категориальных признаков по целевой переменной, важно не использовать оценки из настоящего и будущего. То есть, в нашем примере, при расчете среднего для конкретного объекта нельзя использовать как оценку текущего объекта, так и оценки, которые были поставлены позже. Иначе возникнет переобучение.
2. В качестве счётчиков можно рассматривать и другие статистики: число встречаемости данного значения, медиану целевой переменной по объектам с тем же значением данного категориального признака, и т. п.
3. Подобные признаки-счётчики можно считать не только по одному категориальному признаку, как в примере с жанром фильма, но и и по набору из нескольких категориальных признаков, например, по паре (жанр, киноперсоны).
4. Счётчики можно считать не только по целевой переменной, но и относительно других признаков.

**Задание 8. (2.5 балла).**
- Используя исходные данные, создайте выборку с набором признаков-счётчиков.
- На полученной выборке с счетчиками постройте предсказания оценок, используя [xgboost](https://xgboost.readthedocs.io/en/stable/), [catboost](https://catboost.ai/en/docs/) или [lightGBM](https://lightgbm.readthedocs.io/en/latest/) (на ваш выбор). Постарайтесь добиться качества, сравнимого с качеством моделей из предыдущего пункта.

Какие признаки можно добавить:  
1. Средняя оценка актёров, которые играют в фильме.
2. Средняя оценка, которую ставит пользователь.
3. Средняя оценка фильма по пользователям.
4. Средняя оценка оценок жанров у фильма. (Фильм имеет несколько жанкров, поэтому можно посчитать среднюю оценку для каждого жанра поотдельности и потом у одного фильма усреднить эти оценки).
5. Взять топ 1 тэг у фильма по весу тега и посчитать среднее число встречаемости этого тега.

In [226]:
genres_df = pd.read_csv("data/movielens/movie_genres.dat", delimiter="\t", encoding="latin1")
actors_df = pd.read_csv("data/movielens/movie_actors.dat", delimiter="\t", encoding="latin1")
movie_tags_df = pd.read_csv("data/movielens/movie_tags.dat", delimiter="\t", encoding="latin1")

In [ ]:
def add_mean_user_rating(data_to_feature: pd.DataFrame, train_data: pd.DataFrame) -> pd.DataFrame:
    user_mean = train_data.groupby("userID").agg(user_mean_rating=("rating", "mean"))
    return data_to_feature.merge(
        user_mean,
        how="left",
        on="userID", 
    )
    

def add_mean_movie_rating(data_to_feature: pd.DataFrame, train_data: pd.DataFrame) -> pd.DataFrame:
    user_mean = train_data.groupby("movieID").agg(movie_mean_rating=("rating", "mean"))
    return data_to_feature.merge(
        user_mean,
        how="left",
        on="movieID", 
    )


def add_mean_actors_rating(data_to_feature: pd.DataFrame, train_data: pd.DataFrame, actors: pd.DataFrame) -> pd.DataFrame:
    actors_mean = actors.groupby("movieID").agg(mean_actors_rating=("ranking", "mean"))
    return data_to_feature.merge(
        actors_mean,
        how="left",
        on="movieID", 
    )
    

def add_mean_genres_rating(data_to_feature: pd.DataFrame, train_data: pd.DataFrame, genres: pd.DataFrame) -> pd.DataFrame:
    movies_with_genre = train_data.merge(
        genres,
        how="left",
        on="movieID"
    )
    
    genre_mean_rating = movies_with_genre.groupby("genre").agg(
        mean_genre_rating=("rating", "mean")
    )
    
    movie_genres_mean = movies_with_genre.merge(
        genre_mean_rating,
        how="left",
        on="genre"
    ).groupby("movieID").agg(
        mean_genres_rating=("mean_genre_rating", "mean")
    )
    
    return data_to_feature.merge(
        movie_genres_mean,
        how="left",
        on="movieID"
    )


def add_user_rating_count(data_to_feature: pd.DataFrame, train_data: pd.DataFrame) -> pd.DataFrame:
    user_count = train_data.groupby("userID").agg(user_rating_count=("rating", "count"))
    return data_to_feature.merge(
        user_count,
        how="left",
        on="userID",
    )


def add_movie_rating_count(data_to_feature: pd.DataFrame, train_data: pd.DataFrame) -> pd.DataFrame:
    movie_count = train_data.groupby("movieID").agg(movie_rating_count=("rating", "count"))
    return data_to_feature.merge(
        movie_count,
        how="left",
        on="movieID",
    )
    
    

def add_user_rating_std(data_to_feature: pd.DataFrame, train_data: pd.DataFrame) -> pd.DataFrame:
    user_std = train_data.groupby("userID").agg(user_rating_std=("rating", "std"))
    return data_to_feature.merge(
        user_std,
        how="left",
        on="userID",
    )


def add_movie_rating_std(data_to_feature: pd.DataFrame, train_data: pd.DataFrame) -> pd.DataFrame:
    movie_std = train_data.groupby("movieID").agg(movie_rating_std=("rating", "std"))
    return data_to_feature.merge(
        movie_std,
        how="left",
        on="movieID",
    )



def add_user_genre_count(
    data_to_feature: pd.DataFrame,
    train_data: pd.DataFrame,
    genres: pd.DataFrame
) -> pd.DataFrame:
    """
    Для каждой строки (user, movie) считаем:
    сколько раз пользователь встречался с жанрами текущего фильма в train.
    Если у фильма несколько жанров, усредняем по ним.
    """

    train_with_genres = train_data.merge(
        genres,
        how="left",
        on="movieID"
    )

    user_genre_count = train_with_genres.groupby(
        ["userID", "genre"]
    ).size().reset_index(name="user_genre_count")

    base = data_to_feature.copy()
    base["_row_id"] = np.arange(len(base))

    tmp = base[["_row_id", "userID", "movieID"]].merge(
        genres,
        how="left",
        on="movieID"
    )

    tmp = tmp.merge(
        user_genre_count,
        how="left",
        on=["userID", "genre"]
    )

    tmp = tmp.groupby("_row_id", as_index=False).agg(
        user_genre_count=("user_genre_count", "mean")
    )

    return base.merge(
        tmp,
        how="left",
        on="_row_id"
    ).drop(columns=["_row_id"])


In [351]:
def nan_ratio(df):
    n_df = df.shape[0]
    for col in df.columns:
        n_nans = df[df[col].isna()].shape[0]
        print(f"{col}: {n_nans / n_df}")

In [358]:
train_features_df = train_df.copy()
train_features_df = add_movie_rating_count(train_features_df, train_df)
train_features_df = add_user_rating_count(train_features_df, train_df)
train_features_df = add_movie_rating_std(train_features_df, train_df)
train_features_df = add_user_rating_std(train_features_df, train_df)
train_features_df = add_user_genre_count(train_features_df, train_df, genres_df)
train_features_df = add_mean_movie_rating(train_features_df, train_df)

train_features_df = add_mean_user_rating(train_features_df, train_df)
train_features_df = add_mean_actors_rating(train_features_df, train_df, actors_df)
train_features_df = add_mean_genres_rating(train_features_df, train_df, genres_df)


test_features_df = test_df.copy()
test_features_df = add_movie_rating_count(test_features_df, train_df)
test_features_df = add_user_rating_count(test_features_df, train_df)
test_features_df = add_movie_rating_std(test_features_df, train_df)
test_features_df = add_user_rating_std(test_features_df, train_df)
test_features_df = add_user_genre_count(test_features_df, train_df, genres_df)
test_features_df = add_mean_movie_rating(test_features_df, train_df)

test_features_df = add_mean_user_rating(test_features_df, train_df)
test_features_df = add_mean_actors_rating(test_features_df, train_df, actors_df)
test_features_df = add_mean_genres_rating(test_features_df, train_df, genres_df)


train_features_df = train_features_df.drop(columns=["userID", "movieID"])
test_features_df = test_features_df.drop(columns=["userID", "movieID"])


fill_values = {
    "user_rating_count": 0,
    "movie_rating_count": 0,
    "user_rating_std": train_df["rating"].std(),
    "movie_rating_std": train_df["rating"].std(),
    "user_genre_mean_rating": train_df["rating"].mean(),
    "user_genre_count": 0,
    "movie_mean_rating": train_df["rating"].mean(),
    "mean_actors_rating": actors_df["ranking"].mean(),
    "user_mean_rating": train_df["rating"].mean(),
    "mean_genres_rating": train_df["rating"].mean()
    
}
train_features_df = train_features_df.fillna(fill_values)
test_features_df = test_features_df.fillna(fill_values)

In [359]:
train_features_df.head()

,rating,movie_rating_count,user_rating_count,movie_rating_std,user_rating_std,user_genre_count,movie_mean_rating,user_mean_rating,mean_actors_rating,mean_genres_rating
0,1.0,192,55,0.956843,1.0795,9.500000,2.807292,3.463636,8.5,3.397928
1,4.5,759,55,0.742175,1.0795,20.500000,4.025033,3.463636,6.0,3.356049
2,4.0,818,55,0.904550,1.0795,20.666667,3.855746,3.463636,23.0,3.524015
3,2.0,233,55,0.948952,1.0795,19.750000,2.197425,3.463636,15.5,3.402096
4,4.0,348,55,0.884700,1.0795,23.333333,3.318966,3.463636,7.5,3.393992


In [360]:
train_features_df.isna().any()

rating                False
movie_rating_count    False
user_rating_count     False
movie_rating_std      False
user_rating_std       False
user_genre_count      False
movie_mean_rating     False
user_mean_rating      False
mean_actors_rating    False
mean_genres_rating    False
dtype: bool

In [361]:
nan_ratio(train_features_df)

rating: 0.0
movie_rating_count: 0.0
user_rating_count: 0.0
movie_rating_std: 0.0
user_rating_std: 0.0
user_genre_count: 0.0
movie_mean_rating: 0.0
user_mean_rating: 0.0
mean_actors_rating: 0.0
mean_genres_rating: 0.0


In [362]:
test_features_df.head()

,rating,movie_rating_count,user_rating_count,movie_rating_std,user_rating_std,user_genre_count,movie_mean_rating,user_mean_rating,mean_actors_rating,mean_genres_rating
0,4.5,239.0,437.0,0.952140,0.699078,91.400000,3.820084,4.125858,11.0,3.464755
1,4.0,503.0,437.0,0.797600,0.699078,123.333333,4.047714,4.125858,15.5,3.529927
2,4.5,248.0,437.0,0.852764,0.699078,56.500000,3.824597,4.125858,4.5,3.411439
3,4.5,197.0,437.0,0.900026,0.699078,162.333333,3.939086,4.125858,14.5,3.464668
4,3.0,30.0,437.0,1.026796,0.699078,135.666667,3.150000,4.125858,17.5,3.431088


In [363]:
test_features_df.isna().any()

rating                False
movie_rating_count    False
user_rating_count     False
movie_rating_std      False
user_rating_std       False
user_genre_count      False
movie_mean_rating     False
user_mean_rating      False
mean_actors_rating    False
mean_genres_rating    False
dtype: bool

In [364]:
nan_ratio(test_features_df)

rating: 0.0
movie_rating_count: 0.0
user_rating_count: 0.0
movie_rating_std: 0.0
user_rating_std: 0.0
user_genre_count: 0.0
movie_mean_rating: 0.0
user_mean_rating: 0.0
mean_actors_rating: 0.0
mean_genres_rating: 0.0


Обучим ``CatBoost``

In [365]:
from catboost import CatBoostRegressor, Pool


target_col = "rating"

X_train = train_features_df.drop(columns=[target_col])
y_train = train_features_df[target_col]

X_test = test_features_df.drop(columns=[target_col], errors="ignore")

train_pool = Pool(
    data=X_train,
    label=y_train
)

test_pool = Pool(
    data=X_test
)


In [366]:
model = CatBoostRegressor(
    loss_function="RMSE",
    iterations=2000,
    learning_rate=0.05,
    depth=8,
    verbose=200
)

model.fit(train_pool)

0:	learn: 0.9932381	total: 41.6ms	remaining: 1m 23s
200:	learn: 0.7874618	total: 4.91s	remaining: 44s
400:	learn: 0.7800002	total: 9.68s	remaining: 38.6s
600:	learn: 0.7742938	total: 14.5s	remaining: 33.8s
800:	learn: 0.7699538	total: 19.2s	remaining: 28.8s
1000:	learn: 0.7660599	total: 24s	remaining: 23.9s
1200:	learn: 0.7625822	total: 28.8s	remaining: 19.1s
1400:	learn: 0.7593933	total: 33.7s	remaining: 14.4s
1600:	learn: 0.7564468	total: 42.2s	remaining: 10.5s
1800:	learn: 0.7536339	total: 51s	remaining: 5.63s
1999:	learn: 0.7509942	total: 1m 3s	remaining: 0us


CatBoostRegressor(depth=8, iterations=2000, learning_rate=0.05, loss_function='RMSE', verbose=200)

In [367]:
pred = model.predict(test_pool)

In [368]:
test_df_with_pred = test_df.copy().reset_index(drop=True)
test_df_with_pred["pred"] = model.predict(test_pool)

In [369]:
rmse = compute_rmse(test_df_with_pred)
map_ = compute_map_with_unknowns(train_df, test_df_with_pred)

print(f"RMSE catboost: {rmse}")
print(f"MAP catboost: {map_}")

RMSE catboost: 0.9209276834266122
MAP catboost: 0.022882330526258415


In [374]:
print(f"{'feature':^25} {'importance':^12}")
for feature, importance in zip(X_train.columns, model.get_feature_importance()):
    print(f"{feature:^25}: {importance:^12.4f}")
    

         feature           importance 
   movie_rating_count    :    4.5763   
    user_rating_count    :   10.2297   
    movie_rating_std     :    3.3398   
     user_rating_std     :   13.2228   
    user_genre_count     :    6.6323   
    movie_mean_rating    :   30.6229   
    user_mean_rating     :   24.8832   
   mean_actors_rating    :    1.6620   
   mean_genres_rating    :    4.8309   


- Какие признаки-счётчики оказались наиболее удачными? Почему? 

Наиболее полезными оказались счётчики, которые напрямую связаны со средним рейтингом айтема или со средними оценками юзера.  
Это достаточно логично - наша модель выучила, что если у фильма хороший средний рейтинг по пользователям, то новым пользоваетлям его тоже стоит советовать. Аналогично со средней оценкой пользователя, если у него она большая, то он склонен зывышать оценки.  

Также можно видеть, что достаточно большой вклад делают фичи связанные с ``std`` пользователей, что тоже логично, так как оно по сути показывает, насколько пользователь "эмоционален" при оценке, после просмотра фильма.  

Кроме этого большой вклад делает фича user_rating_count, которая по сути показывает, насколько хорошо мы знаем пользователя по количеству его оценок (чем меньше он сделал оценок, тем мы меньше полагаемся на его опыт).

__Единственная проблема:__ 

В нашей тестовой выборке очень много пользователей, которых не было в тренировочной выборке, из за этого приходится пропуски, которые возникают при подсчете статистик именно по пользователям (пропусков около 70%), заполнять средними значениями из тренировчной выборки. Как будто бы это очень грубое приближение для новых пользователей, но с дргой стороны, непонятно, что может быть лучше для рекомендации новым пользователям, чем то, что смотрит большинство.

**Задание 9. (0.5 балла).** Приведите сравнение качества всех моделей, использованных в работе, руководствуясь значениями описанных метрик. Какие из моделей оказались лучше других по каждой из метрик? Чем это можно объяснить?

Если сранивать все модели с байзлайном - рекомендации ``most_popular`` - , то можно заметить, что сам байзлайн не такой уж и плохой сам по себе.  

Большинство алгоритмов выше не смогли пробить порог байзлайна по метрике ``MAP`` (которая больше всего отображает качество ранжирования).

Самую хорошую метрику ``MAP`` имеет алгоритм SLIM с бинаризированными данными, и это довольно логично. Суть алгоритма SLIM заключается в линейной модели, которая рекомендует, основываясь на признаках item-item, то есть она выучивает, какие фильмы смотрят чаще всего вместе и рекомендует к конкретному фильму, что на самом деле лучше всего отражает логику подбора фильмов (людям чаще всего нравятся фильмы определённого жанра, поэтому лучше рекомендовать те фильмы, которые больше всего похожи на те, которые они уже смотрели; которые лучше всего подходят в пару к фильмам, которые они уже смотрели).  

Другие алгоритмы связаны либо с матрицчной факторизацией, для хорошей работы которой у нас слигком мало данных - матрицы очень разряженные, либо с поиском по ``train_df`` статистик и последующем предсказанияем по ним, что тоже не очень хорошо работает, из за большого различия между ``train_df`` и ``test_df`` по количеству новых пользователей

### Mixigen

На семинаре рассматривался алгоритм Mixigen-а для замешивания различных результатов кандидатогенераторов в единый список.

Напомним, что mixigen - это алгоритм который максимизируем сумарную вероятность попадания в выдачу айтемов из кандидатогенераторов, при этом формируя список на порядки меньшего размера, чем размер списка из уникальных айтемов со всех кандидатогенераторов. Подробнее про него можно почитать [тут](https://www.linkedin.com/pulse/optimizing-candidate-generation-scale-mixigen-michael-roizner-n49se/).

Но так как у нас система упрощена, у нас есть 2 главных недостатка:
- нет информации о конверсии результатов каждого из кворумов
- формально топ, который мы берем из каждого кандидата, целиком идет в расчет метрики MAP


**Задание 10. (2 балла).** Упрощенный миксиджен

Здесь вам предлагается придумать процедуру замешивания айтемов из всех кандидатогенераторов, отталкиваясь от идеи mixigen, так чтобы при максимизировать метрику MAP при фиксированном размере топа (10/30/50).

Для этого нам нужно взять по топу из каждого кандидатогенератора так, чтобы суммарно у нас было по 10/30/50 айтемов и посчитать на них MAP метрику.

Чтобы это сделать, предлагается посчитать и воспользоваться для замешивания следующими статистиками:
- среднее позапросное количество релевантных пользователю фильмов
- долю релевантных фильмов на конкретной позиции для каждого из кворумов

Первую статистику мы можем использовать сразу для развесовки.

Вторую, с использованием [софтмакса гумбеля](https://habr.com/ru/companies/yandex/articles/834262/), для итеративного построения результирущего списка.

Воспользуйтесь этими стратегиями и посчитайте MAP@K, для K=10,30,50 для каждой из схем и сравните с аналогичными метриками но по каждому из кворумов по отдельности.

## Ранжирование

![](http://i.imgur.com/2QnD2nF.jpg)

Задачу поискового ранжирования можно описать следующим образом: имеется множество документов $d \in D$ и множество запросов $q \in Q$. Требуется оценить *степень релевантности* документа по отношению к запросу: $(q, d) \mapsto r$, относительно которой будет производиться ранжирование. Для восстановления этой зависимости используются методы машинного обучения. Обычно используется три типа:
 - признаки запроса $q$, например: мешок слов текста запроса, его длина, ...
 - документа $d$, например: значение PageRank, мешок слов, доменное имя, ...
 - пары $(q, d)$, например: число вхождений фразы из запроса $q$ в документе $d$, ...

Одна из отличительных особенностей задачи ранжирования от классических задач машинного обучения заключается в том, что качество результата зависит не от предсказанных оценок релевантности, а от порядка следования документов в рамках конкретного запроса, т.е. важно не абсолютное значение релевантности (его достаточно трудно формализовать в виде числа), а то, более или менее релевантен документ относительно других документов.


### Подходы к решению задачи ранжирования
Существуют 3 основных подхода, различие между которыми в используемой функции потерь:
  
1. **Pointwise подход**. В этом случае рассматривается *один объект* (в случае поискового ранжирования - конкретный документ) и функция потерь считается только по нему. Любой стандартный классификатор или регрессор может решать pointwise задачу ранжирования, обучившись предсказывать значение таргета. Итоговое ранжирование получается после сортировки документов к одному запросу по предсказанию такой модели.
2. **Pairwise подход**. В рамках данной модели функция потерь вычисляется по *паре объектов*. Другими словами, функция потерь штрафует модель, если отражированная этой моделью пара документов оказалась в неправильном порядке.
3. **Listwise подход**. Этот подход использует все объекты для вычисления функции потерь, стараясь явно оптимизировать правильный порядок.

### Оценка качества

Для оценивания качества ранжирования найденных документов в поиске используются асессорские оценки. Само оценивание происходит на скрытых от обучения запросах $Queries$. Для этого традиционно используется метрика *DCG* ([Discounted Cumulative Gain](https://en.wikipedia.org/wiki/Discounted_cumulative_gain)) и ее нормализованный вариант — *nDCG*, всегда принимающий значения от 0 до 1.
Для одного запроса DCG считается следующим образом:
$$ DCG = \sum_{i=1}^P\frac{(2^{rel_i} - 1)}{\log_2(i+1)}, $$

где $P$ — число документов в поисковой выдаче, $rel_i$ — релевантность (асессорская оценка) документа, находящегося на i-той позиции.

*IDCG* — идеальное (наибольшее из возможных) значение *DCG*, может быть получено путем ранжирования документов по убыванию асессорских оценок.

Итоговая формула для расчета *nDCG*:

$$nDCG = \frac{DCG}{IDCG} \in [0, 1].$$

Чтобы оценить значение *nDCG* на выборке $Queries$ ($nDCG_{Queries}$) размера $N$, необходимо усреднить значение *nDCG* по всем запросам  выборки:
$$nDCG_{Queries} = \frac{1}{N}\sum_{q \in Queries}nDCG(q).$$

### Данные

Данные доступны [здесь](https://disk.yandex.ru/d/8nLlDeUFpWHOOw)


В рамках нашей задачи «документом» будет являться организация.

Разбейте обучающую выборку на обучение и контроль в соотношении 70 / 30. Обратите внимание, что разбивать необходимо множество запросов, а не строчки датасета.


In [3]:
DATA_FILE_RANK = "ML2_2023_lab4_ranking\\train.csv" if os.name == "nt" else "data/ML2_2023_lab4_ranking/train.csv"

dataset = pd.read_csv(DATA_FILE_RANK)


rows_count = dataset.shape[0]
train_rows_target = int(rows_count * 0.7)

query_sizes = (
    dataset.groupby("query")
    .size()
    .reset_index(name="cnt_query")
)

rng = np.random.default_rng(42)
query_sizes = query_sizes.sample(frac=1, random_state=42).reset_index(drop=True)

train_queries = []
cnt_sum = 0
for _, row in query_sizes.iterrows():
    cur_query = row["query"]
    cnt = row["cnt_query"]
    if cnt_sum >= train_rows_target:
        break

    train_queries.append(cur_query)
    cnt_sum += cnt

train_df = dataset[dataset["query"].isin(train_queries)].copy()
test_df = dataset[~dataset["query"].isin(train_queries)].copy()

print("Total rows:", len(dataset))
print("Train rows:", len(train_df))
print("Test rows:", len(test_df))
print("Train ratio:", len(train_df) / len(dataset))
print("Queries in train:", train_df["query"].nunique())
print("Queries in test:", test_df["query"].nunique())

Total rows: 29274
Train rows: 20519
Test rows: 8755
Train ratio: 0.7009291521486644
Queries in train: 1039
Queries in test: 453


In [4]:
train_df.isna().any()

query_id         False
query            False
region           False
org_name         False
org_id           False
window_center    False
window_size      False
relevance        False
dtype: bool

In [5]:
train_df.head()

,query_id,query,region,org_name,org_id,window_center,window_size,relevance
0,11,"суд, Украина, Днепропетровская область, Днепро...",21775,Суд Жовтневого району міста Дніпропетровськ,1021049127,"34.613119,48.506531","0.025928,0.017380",0.0
1,11,"суд, Украина, Днепропетровская область, Днепро...",21775,Дніпропетровський окружний адміністративний суд,1602348889,"34.613119,48.506531","0.025928,0.017380",0.0
2,11,"суд, Украина, Днепропетровская область, Днепро...",21775,Бабушкінський районний суд,1105837793,"34.613119,48.506531","0.025928,0.017380",0.0
3,11,"суд, Украина, Днепропетровская область, Днепро...",21775,Красногвардійський районний суд,1066267658,"34.613119,48.506531","0.025928,0.017380",0.0
4,11,"суд, Украина, Днепропетровская область, Днепро...",21775,Жовтневий суд,1661586235,"34.613119,48.506531","0.025928,0.017380",0.0


Далее рассмотрим несколько подходов предсказания релевантности. Для оценивания качества моделей используйте метрику nDCG на контроле. В случае подбора гиперпараметров используйте кросс-валидацию по 5 блокам, где разбиение должно быть по запросам, а не строчкам датасета.

Пример подсчета метрики ndcg по сессиям будет представлен ниже:

In [6]:
from sklearn.metrics import ndcg_score
import numpy as np

np.random.seed(43)

def my_cool_ranking_algo(features):
    return np.random.random()


true_labels = [
    np.random.random(size=session_length)
    for session_length in np.random.randint(10, 30, size=40)
]
pred_labels = [
    np.array([my_cool_ranking_algo(...) for _ in range(session_length)])
    for session_length in map(len, true_labels)
]
np.mean([
    ndcg_score(y_true=true_label.reshape(1, -1), y_score=pred_label.reshape(1, -1))
    for true_label, pred_label in zip(true_labels, pred_labels)
])

np.float64(0.8027760862768976)

### Построение обучающего пула


В данных нет фичей, зато есть строковые представления запроса и объекта: отдельной задачей вам нужно представить вектор объекта по текстовому представлению запроса и названии объекта.

**Задание 11. (1 балл)**

В данном пункте вам следует закодировать все объекты набором признаков не подсматривая в тест выборку.


In [ ]:
from sklearn.preprocessing import StandardScaler
from sklearn.metrics.pairwise import cosine_similarity
from sentence_transformers import SentenceTransformer


def normalize_text(s: pd.Series) -> pd.Series:
    return (
        s.fillna("")
         .astype(str)
         .str.lower()
         .str.replace(r"[^a-zа-я0-9\s]+", " ", regex=True)
         .str.replace(r"\s+", " ", regex=True)
         .str.strip()
    )


def build_pair_features_bert(
    df: pd.DataFrame,
    bert_model=None,
    fit: bool = False,
    model_name: str = "sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2",
    device=device
):
    """
    Возвращает dense feature matrix:
        [query_emb | org_emb | cosine_sim | numeric_features]

    fit=True:
        создаёт и загружает BERT-модель
    fit=False:
        использует уже переданную bert_model
    """

    data = df.copy()

    data["query_norm"] = normalize_text(data["query"])
    data["org_name_norm"] = normalize_text(data["org_name"])

    if fit:
        bert_model = SentenceTransformer(model_name, device=device)

    # sentence embeddings
    q_emb = bert_model.encode(
        data["query_norm"].tolist(),
        batch_size=64,
        show_progress_bar=True,
        convert_to_numpy=True,
        normalize_embeddings=True,
    )

    o_emb = bert_model.encode(
        data["org_name_norm"].tolist(),
        batch_size=64,
        show_progress_bar=True,
        convert_to_numpy=True,
        normalize_embeddings=True,
    )

    # cosine similarity между query и org_name
    cos_sim = np.sum(q_emb * o_emb, axis=1, keepdims=True).astype(np.float32)

    # простые текстовые и числовые признаки
    query_tokens = data["query_norm"].str.split()
    org_tokens = data["org_name_norm"].str.split()

    common_count = []
    jaccard = []
    query_len = []
    org_len = []
    query_word_count = []
    org_word_count = []
    query_in_org = []
    org_in_query = []

    for q, o in zip(query_tokens, org_tokens):
        q_set = set(q)
        o_set = set(o)

        inter = len(q_set & o_set)
        union = len(q_set | o_set)

        common_count.append(inter)
        jaccard.append(inter / union if union > 0 else 0.0)
        query_len.append(len(" ".join(q)))
        org_len.append(len(" ".join(o)))
        query_word_count.append(len(q))
        org_word_count.append(len(o))
        query_in_org.append(int(" ".join(q) in " ".join(o)))
        org_in_query.append(int(" ".join(o) in " ".join(q)))

    window = (
        data["window_size"]
        .fillna("0,0")
        .astype(str)
        .str.split(",", expand=True)
        .astype(float)
    )

    num_features = np.column_stack([
        common_count,
        jaccard,
        query_len,
        org_len,
        query_word_count,
        org_word_count,
        query_in_org,
        org_in_query,
        data["region"].to_numpy(dtype=np.float32),
        window[0].to_numpy(dtype=np.float32),
        window[1].to_numpy(dtype=np.float32),
    ]).astype(np.float32)

    # финальные dense features
    X = np.hstack([
        q_emb.astype(np.float32),
        o_emb.astype(np.float32),
        cos_sim,
        num_features,
    ]).astype(np.float32)

    return X, bert_model

In [9]:
X_train, bert_model = build_pair_features_bert(train_df, fit=True)
y_train = train_df["relevance"].to_numpy()

X_test, _ = build_pair_features_bert(test_df, bert_model=bert_model, fit=False)
y_test = test_df["relevance"].to_numpy()

X_train.shape

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Batches:   0%|          | 0/321 [00:00<?, ?it/s]

Batches:   0%|          | 0/321 [00:00<?, ?it/s]

Batches:   0%|          | 0/137 [00:00<?, ?it/s]

Batches:   0%|          | 0/137 [00:00<?, ?it/s]

(20519, 780)

In [11]:
X_train

array([[ 8.1330221e-03,  9.4848737e-02, -2.4778543e-02, ...,
         2.1775000e+04,  2.5928000e-02,  1.7379999e-02],
       [ 8.1330221e-03,  9.4848737e-02, -2.4778543e-02, ...,
         2.1775000e+04,  2.5928000e-02,  1.7379999e-02],
       [ 8.1330221e-03,  9.4848737e-02, -2.4778543e-02, ...,
         2.1775000e+04,  2.5928000e-02,  1.7379999e-02],
       ...,
       [ 7.7058405e-02,  6.7610554e-02, -2.9094227e-02, ...,
         1.0857000e+04,  3.4021440e+00,  1.0631020e+00],
       [ 7.7058405e-02,  6.7610554e-02, -2.9094227e-02, ...,
         1.0857000e+04,  3.4021440e+00,  1.0631020e+00],
       [ 7.7058405e-02,  6.7610554e-02, -2.9094227e-02, ...,
         1.0857000e+04,  3.4021440e+00,  1.0631020e+00]], dtype=float32)

### Ранжируем с LinearRegression

**Задание 12. (1 балла)**

Давайте в качестве бейзлайна воспользуемся линейной регрессией, для обучения в pointwise режиме на MSE от разметки. Затем посмотрим, какой NDCG он нам дает.

In [9]:
from sklearn.linear_model import LinearRegression


model = LinearRegression()
model.fit(X_train, y_train)

,"fit_intercept fit_intercept: bool, default=TrueWhether to calculate the intercept for this model. If setto False, no intercept will be used in calculations(i.e. data is expected to be centered).",True
,"copy_X copy_X: bool, default=TrueIf True, X will be copied; else, it may be overwritten.",True
,"tol tol: float, default=1e-6The precision of the solution (`coef_`) is determined by `tol` whichspecifies a different convergence criterion for the `lsqr` solver.`tol` is set as `atol` and `btol` of :func:`scipy.sparse.linalg.lsqr` whenfitting on sparse training data. This parameter has no effect when fittingon dense data... versionadded:: 1.7",1e-06
,"n_jobs n_jobs: int, default=NoneThe number of jobs to use for the computation. This will only providespeedup in case of sufficiently large problems, that is if firstly`n_targets > 1` and secondly `X` is sparse or if `positive` is setto `True`. ``None`` means 1 unless in a:obj:`joblib.parallel_backend` context. ``-1`` means using allprocessors. See :term:`Glossary ` for more details.",None
,"positive positive: bool, default=FalseWhen set to ``True``, forces the coefficients to be positive. Thisoption is only supported for dense arrays.For a comparison between a linear regression model with positive constraintson the regression coefficients and a linear regression without such constraints,see :ref:`sphx_glr_auto_examples_linear_model_plot_nnls.py`... versionadded:: 0.24",False


In [10]:
test_pred = model.predict(X_test)

test_df_with_pred = test_df.copy()
test_df_with_pred["pred"] = test_pred

In [11]:
test_df_with_pred.head()

,query_id,query,region,org_name,org_id,window_center,window_size,relevance,pred
434,258,фастфуд,10365,Експрес-кафе Гном,1142145026,"25.991333,48.296761","0.425118,0.394303",0.14,0.097600
435,258,фастфуд,10365,Кіоск Шаурма,1555753904,"25.991333,48.296761","0.425118,0.394303",0.14,0.103653
436,258,фастфуд,10365,Форнеттi,1776339335,"25.991333,48.296761","0.425118,0.394303",0.14,0.099109
437,258,фастфуд,10365,Форнеттi,1769224328,"25.991333,48.296761","0.425118,0.394303",0.14,0.099109
438,258,фастфуд,10365,Форнеттi,1756533297,"25.991333,48.296761","0.425118,0.394303",0.14,0.099109


In [12]:
def compute_mean_ndcg(
    df: pd.DataFrame, 
    query_col: str, 
    target_col: str, 
    pred_col: str
) -> float:
    
    ndcgs = []
    for _, group in df.groupby(query_col):
        if len(group) < 2:
            continue
        
        y_true = group[target_col].to_numpy().reshape(1, -1)
        y_score = group[pred_col].to_numpy().reshape(1, -1)

        ndcg = ndcg_score(y_true, y_score)
        ndcgs.append(ndcg)

    return float(np.mean(ndcgs)) if ndcgs else 0.0

In [13]:
ndcg = compute_mean_ndcg(
    test_df_with_pred,
    query_col="query",
    target_col="relevance",
    pred_col="pred"
)

print("NDCG LinearReg:", ndcg)

NDCG LinearReg: 0.6728925941931171


### Ранжируем с торчем

Бейзлайн у нас есть, давайте теперь улучшим наши предсказания с помощью pairwise режима обучения.

**Задание 13. (2 балла)**

Реализуйте Rank-net с батчевым обновлением на обучающей выборке. В качестве модели можете взять как MLP, так и более сложную модельку - хоть предобученные енкодеры с MLP головой. Ваша задача побить по NDCG бейзлайн из линейной регрессии.



In [14]:
import torch.nn as nn
import torch.nn.functional as F


class RankNetMLP(nn.Module):
    def __init__(self, input_dim: int, hidden_dims: tuple = (256, 128, 64), p_dropout=0.2):
        super().__init__()
        
        dims = [input_dim] + list(hidden_dims)
        layers = []
        
        for in_dim, out_dim in zip(dims[:-1], dims[1:]):
            layers.extend([
                nn.Linear(in_dim, out_dim),
                nn.LayerNorm(out_dim),
                nn.ReLU(),
                nn.Dropout(p_dropout)
            ])
            
        layers.append(nn.Linear(dims[-1], 1))
        self.net = nn.Sequential(*layers)
    
    
    def forward(self, x):
        return self.net(x).squeeze(-1)

In [15]:
from torch.utils.data import Dataset, DataLoader


def split_by_query_id(df: pd.DataFrame, val_frac: float = 0.1, seed: int = 42):
    queries = df["query_id"].drop_duplicates().sample(frac=1.0, random_state=seed).to_list()
    n_val = max(1, int(len(queries) * val_frac))
    val_queries = set(queries[:n_val])

    train_part = df[~df["query_id"].isin(val_queries)].copy()
    val_part = df[df["query_id"].isin(val_queries)].copy()

    return train_part, val_part


class PairwiseQueryDataset(Dataset):
    def __init__(self, 
                X: np.ndarray,
                y: np.ndarray,
                query_ids: np.ndarray,
                 
                seed: int = 43,
                pairs_per_query: int = 20,
                q_return: bool = False
    ):
        super().__init__()
        
        self.X: sparse.csr_matrix = X
        self.y = y.astype(np.float32).round(5)
        self.query_ids = np.asarray(query_ids)
        self.rng = np.random.default_rng(seed)
        self.pairs_per_query = pairs_per_query
        self.q_return = q_return
        
        self.query_to_buckets = {}
        self.queries = []
        for q_id in np.unique(self.query_ids):
            idxs = np.where(self.query_ids == q_id)[0]
            
            rels = self.y[idxs]
            uniq_rels = np.unique(rels)
            
            if len(uniq_rels) < 2:
                continue
            
            rel_dict = {uniq_rel: idxs[uniq_rel == rels] for uniq_rel in uniq_rels}
            self.query_to_buckets[q_id] = rel_dict
            self.queries.append(q_id)
        
        self.queries = np.array(self.queries)
        
    
    def __len__(self):
        return len(self.queries) * self.pairs_per_query
    
    
    def _row_to_tensor(self, idx):
        row = self.X[idx].toarray().ravel().astype(np.float32)
        return torch.from_numpy(row)
    
    
    def __getitem__(self, index):
        q = self.queries[index % self.queries.shape[0]]
        
        buckets = self.query_to_buckets[q]
        levels = np.array(sorted(buckets.keys()), dtype=np.float32)
        
        li, lj = self.rng.choice(len(levels), size=2, replace=False)
        
        max_rel = max(levels[li], levels[lj])
        min_rel = min(levels[li], levels[lj])
        
        pos_idx = self.rng.choice(buckets[max_rel])
        neg_idx = self.rng.choice(buckets[min_rel])
        
        x_pos = self._row_to_tensor(pos_idx)
        x_neg = self._row_to_tensor(neg_idx)
        
        
        return x_pos, x_neg


In [16]:
@torch.no_grad()
def predict_scores(model: nn.Module, X: sparse.csr_matrix, batch_size: int = 4096) -> np.ndarray:
    device = next(model.parameters()).device

    model.eval()
    preds = []

    for start in range(0, X.shape[0], batch_size):
        batch = X[start:start + batch_size]
        xb = torch.from_numpy(batch.toarray().astype(np.float32)).to(device)
        sb = model(xb)
        preds.append(sb.detach().cpu().numpy())

    return np.concatenate(preds, axis=0)



def train_loop(model,
                  train_loader: DataLoader,
                  X_val: np.ndarray,
                  val_df: pd.DataFrame,
                  device: str,
                  epochs: int = 12,
                  lr: float = 1e-3,
                  weight_decay: float = 1e-4,
                  patience: int = 3,
                  grad_clip: float = 1.0
):
    
    optimizer = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=weight_decay)

    best_state = copy.deepcopy(model.state_dict())
    best_ndcg = -1.0
    bad_epochs = 0

    for epoch in range(1, epochs + 1):
        model.train()
        total_loss = 0.0
        n_batches = 0
        
        
        for x_pos, x_neg in train_loader:
            x_pos = x_pos.to(device)
            x_neg = x_neg.to(device)
            w = w.to(device)

            s_pos = model(x_pos)
            s_neg = model(x_neg)

            # RankNet pairwise loss:
            # L = w_ij * log(1 + exp(-(s_pos - s_neg)))
            loss = (F.softplus(-(s_pos - s_neg))).mean()

            optimizer.zero_grad(set_to_none=True)
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), grad_clip)
            optimizer.step()

            total_loss += float(loss.item())
            n_batches += 1


        val_pred = predict_scores(model, X_val)
        val_scored = val_df.copy()
        val_scored["pred"] = val_pred
        val_ndcg = compute_mean_ndcg(
            val_scored,
            query_col="query_id",
            target_col="relevance",
            pred_col="pred"
        )

        avg_loss = total_loss / max(1, n_batches)
        print(f"Epoch {epoch:02d} | loss={avg_loss:.5f} | val NDCG={val_ndcg:.5f}")



        if val_ndcg > best_ndcg + 1e-5:
            best_ndcg = val_ndcg
            best_state = copy.deepcopy(model.state_dict())
            bad_epochs = 0
        else:
            bad_epochs += 1
            if bad_epochs >= patience:
                print("Early stopping.")
                break


    model.load_state_dict(best_state)
    return model, best_ndcg


Добавим валидационную выборку

In [56]:
train_sub_df, val_df = split_by_query_id(train_df, val_frac=0.1, seed=42)

X_train, q_vec, o_vec = build_pair_features(train_sub_df, fit=True)
y_train = train_sub_df["relevance"].to_numpy()

X_val, _, _ = build_pair_features(val_df, query_vectorizer=q_vec, org_vectorizer=o_vec, fit=False)
y_val = val_df["relevance"].to_numpy()

X_test, _, _ = build_pair_features(test_df, query_vectorizer=q_vec, org_vectorizer=o_vec, fit=False)
y_test = test_df["relevance"].to_numpy()

KeyError: 'window_size'

In [18]:
train_dataset = PairwiseQueryDataset(
    X=X_train,
    y=y_train,
    query_ids=train_sub_df["query_id"].to_numpy(),
    pairs_per_query=24,
    seed=42
)

train_loader = DataLoader(
    train_dataset,
    batch_size=256,
    shuffle=True,
    num_workers=0,
    drop_last=True
)


In [48]:
model = RankNetMLP(input_dim=X_train.shape[1], hidden_dims=(256, 128, 64), p_dropout=0.2).to(device)

model, best_val_ndcg = train_loop(
    model=model,
    train_loader=train_loader,
    X_val=X_val,
    val_df=val_df,
    device=device,
    epochs=15,
    lr=1e-3,
    weight_decay=1e-4,
    patience=3,
    grad_clip=1.0
)

Epoch 01 | loss=0.61554 | val NDCG=0.75517
Epoch 02 | loss=0.45098 | val NDCG=0.74422
Epoch 03 | loss=0.39970 | val NDCG=0.76598
Epoch 04 | loss=0.37129 | val NDCG=0.75178
Epoch 05 | loss=0.35323 | val NDCG=0.74596
Epoch 06 | loss=0.33906 | val NDCG=0.74804
Early stopping.


In [49]:
test_pred = predict_scores(model, X_test)
test_df_with_pred = test_df.copy()
test_df_with_pred["pred"] = test_pred

test_ndcg = compute_mean_ndcg(
    test_df_with_pred,
    query_col="query_id",
    target_col="relevance",
    pred_col="pred"
)

print("RankNet")
print("Best val NDCG:", best_val_ndcg)
print("Test NDCG:", test_ndcg)

Best val NDCG: 0.7659808067411567
Test NDCG: 0.7061520182551481


**Задание 14. (1 балл)**

Теперь реализуйте Lambda-rank с использованием NDCG для развесовки. Ваша задача получить качество не хуже, чем у алгоритма из задания 16.

Идея ``LambdaRank`` - добавить в лосс вес, который зависит от конкретной пары объектов $i$ и $j$. Чем больше изменится в метрика NDCG от перенстановки, тем сильнее мы учитываем в лосее разницу между ранками на этих объектах.  

Поэтому ниже я реализовал градиентный спуск, который считается не по сулчайным парам внтури одного запроса, а по всем парам объектов нутри одного запроса, чтобы считать разницу $\Delta \text{NDCG}$ и использовать его как коэффициент перед лоссом, благодаря этому мы учитываем нашу метрику в лоссе.

In [19]:
class LambdaRankQueryDataset(Dataset):
    def __init__(
        self,
        X: sparse.csr_matrix,
        y: np.ndarray,
        query_ids: np.ndarray,
    ):
        super().__init__()

        self.X = X
        self.y = y.astype(np.float32)
        self.query_ids = np.asarray(query_ids)

        self.unique_queries = np.unique(self.query_ids)

        self.query_to_indices = {}

        for q in self.unique_queries:
            idxs = np.where(self.query_ids == q)[0]
            if len(idxs) < 2:
                continue

            self.query_to_indices[q] = idxs

        self.unique_queries = np.array(
            list(self.query_to_indices.keys())
        )

    def __len__(self):
        return len(self.unique_queries)

    def __getitem__(self, idx):
        q = self.unique_queries[idx]

        doc_idxs = self.query_to_indices[q]

        X_q = self.X[doc_idxs].toarray().astype(np.float32)
        y_q = self.y[doc_idxs].astype(np.float32)

        return (
            torch.from_numpy(X_q),
            torch.from_numpy(y_q)
        )

In [20]:
def dcg_torch(rels: torch.Tensor):
    """
    rels: [n_docs]
    """
    device = rels.device

    positions = torch.arange(
        1,
        rels.shape[0] + 1,
        device=device,
        dtype=torch.float32
    )
    discounts = 1.0 / torch.log2(positions + 1.0)
    gains = torch.pow(2.0, rels) - 1.0

    return torch.sum(gains * discounts)


def compute_delta_ndcg(
    y_true: torch.Tensor,
    scores: torch.Tensor,
):
    """
    Возвращает матрицу:
        delta_ndcg[i, j]

    насколько изменится NDCG,
    если поменять местами документы i и j
    """

    device = y_true.device
    n = y_true.shape[0]

    # текущий ranking модели
    order = torch.argsort(scores, descending=True)

    rels_sorted = y_true[order]

    ideal_order = torch.argsort(y_true, descending=True)
    ideal_rels = y_true[ideal_order]

    idcg = dcg_torch(ideal_rels)

    if idcg <= 0:
        return torch.zeros((n, n), device=device)

    positions = torch.arange(
        1,
        n + 1,
        device=device,
        dtype=torch.float32
    )

    discounts = 1.0 / torch.log2(positions + 1.0)
    gains = torch.pow(2.0, rels_sorted) - 1.0
    delta = torch.zeros((n, n), device=device)

    for i in range(n):
        for j in range(n):

            if rels_sorted[i] == rels_sorted[j]:
                continue

            old = (
                gains[i] * discounts[i]
                + gains[j] * discounts[j]
            )

            new = (
                gains[i] * discounts[j]
                + gains[j] * discounts[i]
            )

            delta_ij = torch.abs(new - old) / idcg

            delta[i, j] = delta_ij

    inv_order = torch.empty_like(order)
    inv_order[order] = torch.arange(n, device=device)

    delta_original = delta[
        inv_order][:, inv_order]

    return delta_original

In [21]:
def train_loop_lambdarank(
    model,
    train_loader,
    X_val,
    val_df,
    device,

    epochs=10,
    lr=1e-3,
    weight_decay=1e-4,
    patience=3,
    grad_clip=1.0
):
    optimizer = torch.optim.AdamW(
        model.parameters(),
        lr=lr,
        weight_decay=weight_decay
    )

    best_state = copy.deepcopy(model.state_dict())
    best_ndcg = -1.0
    bad_epochs = 0

    for epoch in range(1, epochs + 1):

        model.train()

        total_loss = 0.0
        n_batches = 0

        for X_q, y_q in tqdm(train_loader):

            X_q = X_q.squeeze(0).to(device)
            y_q = y_q.squeeze(0).to(device)

            if X_q.shape[0] < 2:
                continue

            scores = model(X_q)

            delta_ndcg = compute_delta_ndcg(
                y_true=y_q,
                scores=scores.detach()
            )

            pair_losses = []
            n_docs = X_q.shape[0]
            for i in range(n_docs):
                for j in range(n_docs):
                    if y_q[i] <= y_q[j]:
                        continue

                    s_ij = scores[i] - scores[j]
                    lambda_ij = delta_ndcg[i, j]

                    pair_loss = (lambda_ij * F.softplus(-s_ij))

                    pair_losses.append(pair_loss)

            if len(pair_losses) == 0:
                continue

            loss = torch.stack(pair_losses).mean()

            optimizer.zero_grad(set_to_none=True)
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), grad_clip)
            optimizer.step()

            total_loss += float(loss.item())
            n_batches += 1



        val_pred = predict_scores(model, X_val)
        val_scored = val_df.copy()
        val_scored["pred"] = val_pred

        val_ndcg = compute_mean_ndcg(
            val_scored,
            query_col="query_id",
            target_col="relevance",
            pred_col="pred"
        )

        avg_loss = total_loss / max(1, n_batches)
        print(f"Epoch {epoch:02d} | loss={avg_loss:.5f} | val NDCG={val_ndcg:.5f}")



        if val_ndcg > best_ndcg + 1e-5:
            best_ndcg = val_ndcg
            best_state = copy.deepcopy(model.state_dict())
            bad_epochs = 0

        else:
            bad_epochs += 1

            if bad_epochs >= patience:
                print("Early stopping.")
                break

    model.load_state_dict(best_state)

    return model, best_ndcg

In [22]:
train_lambda_dataset = LambdaRankQueryDataset(
    X=X_train,
    y=y_train,
    query_ids=train_sub_df["query_id"]
)

train_lambda_loader = DataLoader(
    train_lambda_dataset,
    batch_size=1,
    shuffle=True
)

In [29]:
model = RankNetMLP(input_dim=X_train.shape[1]).to(device)

model, best_ndcg_val = train_loop_lambdarank(
    model=model,
    train_loader=train_lambda_loader,
    X_val=X_val,
    val_df=val_df,
    device=device,
    
    epochs=15,
    lr=1e-3,
    weight_decay=1e-4,
    patience=3
)

  0%|          | 0/1141 [00:00<?, ?it/s]

Epoch 01 | loss=0.07143 | val NDCG=0.74064


  0%|          | 0/1141 [00:00<?, ?it/s]

Epoch 02 | loss=0.07368 | val NDCG=0.74650


  0%|          | 0/1141 [00:00<?, ?it/s]

...

Epoch 05 | loss=0.06582 | val NDCG=0.75959


  0%|          | 0/1141 [00:00<?, ?it/s]

Epoch 06 | loss=0.05702 | val NDCG=0.76467


  0%|          | 0/1141 [00:00<?, ?it/s]

Epoch 07 | loss=0.05627 | val NDCG=0.74438
Early stopping.


In [31]:
test_pred = predict_scores(model, X_test)
test_df_with_pred = test_df.copy()
test_df_with_pred["pred"] = test_pred

test_ndcg = compute_mean_ndcg(
    test_df_with_pred,
    query_col="query_id",
    target_col="relevance",
    pred_col="pred"
)

print("LambdaRank")
print("Best val NDCG:", best_ndcg_val)
print("Test NDCG:", test_ndcg)

LambdaRank
Best val NDCG: 0.7720145264710192
Test NDCG: 0.7212593875328396


**Задание 15. (1 балл)**

А теперь вспомним о предыдущей части лабы, а именно про LightFM и WARP лосс. Добавьте идею с семплированием негатива в ваш алгоритм и оцените получаемое качество алгоритма.

Идея ``WARP`` лосса:  
Считаем его так же для всех элементов одного запроса. Для каждого объекта ``s_pos`` мы находим такой объект ``s_neg``, для которых    
``s_pos >= s_neg + margin``.

И, именно на такой паре мы считаем лосс из RankNet и домножаем его на warp-коэффициент, который показывает, как долго мы внутри одного запроса семлированием искали объект, который наша модель неправильно ранжирует. Чем дольше мы искали подходящий негатинвный объект, тем вес этой пары меньше в нашем лоссе. То есть WARP концентриутеся на "очень плохих" парах примерах.

In [ ]:
def train_loop_warp(
    model,
    train_loader,
    X_val,
    val_df,
    device,

    epochs=10,
    lr=1e-3,
    weight_decay=1e-4,
    patience=3,
    grad_clip=1.0,

    margin=1.0,
    max_negative_trials=50
):

    optimizer = torch.optim.AdamW(
        model.parameters(),
        lr=lr,
        weight_decay=weight_decay
    )

    best_state = copy.deepcopy(model.state_dict())
    best_ndcg = -1.0
    bad_epochs = 0

    for epoch in range(1, epochs + 1):

        model.train()

        total_loss = 0.0
        n_batches = 0
        for X_q, y_q in tqdm(train_loader):
            X_q = X_q.squeeze(0).to(device)
            y_q = y_q.squeeze(0).to(device)

            n_docs = X_q.shape[0]
            if n_docs < 2:
                continue

            scores = model(X_q)
            pair_losses = []
            for pos_idx in range(n_docs):

                pos_rel = y_q[pos_idx]
                neg_candidates = torch.where(y_q < pos_rel)[0]
                if len(neg_candidates) == 0:
                    continue

                s_pos = scores[pos_idx]
                for trial in range(1, max_negative_trials + 1):
                    rand_id = torch.randint(0, len(neg_candidates), (1,), device=device).item()
                    neg_idx = neg_candidates[rand_id]
                    s_neg = scores[neg_idx]

                    violation = margin - (s_pos - s_neg)

                    if violation > 0:
                        approx_rank = (n_docs - 1) / trial
                        warp_weight = math.log1p(approx_rank)
                        loss_ij = warp_weight * F.relu(violation)
                        pair_losses.append(loss_ij)

                        break

                # если hard negative не нашли,
                # query уже хорошо отсортирован

            if len(pair_losses) == 0:
                continue

            loss = torch.stack(pair_losses).mean()
            optimizer.zero_grad(set_to_none=True)
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), grad_clip)
            optimizer.step()

            total_loss += float(loss.item())
            n_batches += 1

    
        val_pred = predict_scores(model, X_val)

        val_scored = val_df.copy()
        val_scored["pred"] = val_pred

        val_ndcg = compute_mean_ndcg(
            val_scored,
            query_col="query_id",
            target_col="relevance",
            pred_col="pred"
        )

        avg_loss = total_loss / max(1, n_batches)
        print(f"Epoch {epoch:02d} | loss={avg_loss:.5f} | val NDCG={val_ndcg:.5f}")


        if val_ndcg > best_ndcg + 1e-5:
            best_ndcg = val_ndcg
            best_state = copy.deepcopy(model.state_dict())
            bad_epochs = 0

        else:
            bad_epochs += 1
            if bad_epochs >= patience:
                print("Early stopping.")
                break

    model.load_state_dict(best_state)

    return model, best_ndcg

In [35]:
model = RankNetMLP(input_dim=X_train.shape[1]).to(device)

model, best_ndcg_val = train_loop_warp(
    model=model,
    train_loader=train_lambda_loader,
    X_val=X_val,
    val_df=val_df,
    device=device,
    
    epochs=10,
    lr=1e-3,
    weight_decay=1e-4,
    patience=3,
    
    margin=1.0,
    max_negative_trials=50
)

  0%|          | 0/1141 [00:00<?, ?it/s]

Epoch 01 | loss=2.71294 | val NDCG=0.73585


  0%|          | 0/1141 [00:00<?, ?it/s]

Epoch 02 | loss=2.60998 | val NDCG=0.73564


  0%|          | 0/1141 [00:00<?, ?it/s]

...

Epoch 08 | loss=2.29130 | val NDCG=0.76920


  0%|          | 0/1141 [00:00<?, ?it/s]

Epoch 09 | loss=2.27834 | val NDCG=0.76646


  0%|          | 0/1141 [00:00<?, ?it/s]

Epoch 10 | loss=2.31374 | val NDCG=0.76599


In [36]:
test_pred = predict_scores(model, X_test)
test_df_with_pred = test_df.copy()
test_df_with_pred["pred"] = test_pred

test_ndcg = compute_mean_ndcg(
    test_df_with_pred,
    query_col="query_id",
    target_col="relevance",
    pred_col="pred"
)

print("WARP-Loss")
print("Best val NDCG:", best_ndcg_val)
print("Test NDCG:", test_ndcg)

WARP-Loss
Best val NDCG: 0.7691967046660632
Test NDCG: 0.7087381491511597


###  Ранжируем с CatBoost

CatBoost имеют несколько функций потерь/метрик для решения задачи ранжирования. Подробный их список с пометкой о возможности оптимизации указан в документации: https://catboost.ai/docs/en/concepts/loss-functions-ranking#usage-information

**Задание 16. (1.5 балла).** Попробуйте различные функции потерь (регрессионные () и ранжирующие), но не меньше 3, для модели CatBoost. Настройте основные параметры моделей (глубина, кол-во деревьев, скорость обучения, регуляризация).

Ваша задача получить сетап, который лучше отработает, чем ваши торчевые реализации 🤠

Сравните построенные модели с точки зрения метрики nDCG на контроле и проанализируйте полученные результаты:
  - какая модель работает лучше всего для данной задачи?
  - в чем достоинства/недостатки каждой?
  - сравните модели между собой:
   - получается ли сравнимое качество линейного pointwise подхода с остальными моделями?
   - заметна ли разница в качестве при использовании бустинга с разными функциями потерь?

In [25]:
train_df.head()

,query_id,query,region,org_name,org_id,window_center,window_size,relevance
0,11,"суд, Украина, Днепропетровская область, Днепро...",21775,Суд Жовтневого району міста Дніпропетровськ,1021049127,"34.613119,48.506531","0.025928,0.017380",0.0
1,11,"суд, Украина, Днепропетровская область, Днепро...",21775,Дніпропетровський окружний адміністративний суд,1602348889,"34.613119,48.506531","0.025928,0.017380",0.0
2,11,"суд, Украина, Днепропетровская область, Днепро...",21775,Бабушкінський районний суд,1105837793,"34.613119,48.506531","0.025928,0.017380",0.0
3,11,"суд, Украина, Днепропетровская область, Днепро...",21775,Красногвардійський районний суд,1066267658,"34.613119,48.506531","0.025928,0.017380",0.0
4,11,"суд, Украина, Днепропетровская область, Днепро...",21775,Жовтневий суд,1661586235,"34.613119,48.506531","0.025928,0.017380",0.0


In [88]:
def preprocess_data(df: pd.DataFrame):
    data = df.copy()
    
    center = data["window_center"].astype(str).str.split(",", expand=True)
    size = data["window_size"].astype(str).str.split(",", expand=True)
    
    data["window_lon"] = center[0].astype(float)
    data["window_lat"] = center[1].astype(float)
    
    data["window_size_w"] = size[0].astype(float)
    data["window_size_h"] = size[1].astype(float)
    
    data["window_area"] = data["window_size_w"] * data["window_size_h"]
    data["window_aspect"] = data["window_size_w"] / (data["window_size_h"] + 1e-8)
    
    data = data.drop(columns=["window_center", "window_size", "query", "org_name"])
    
    return data


def drop_trivial_queries(
    df: pd.DataFrame,
    group_col: str = "query_id",
    target_col: str = "relevance",
) -> pd.DataFrame:
    stats = df.groupby(group_col)[target_col].agg(["size", "nunique"])
    good_queries = stats[(stats["size"] >= 2) & (stats["nunique"] >= 2)].index
    return df[df[group_col].isin(good_queries)].copy()




train_sub_df, val_df = split_by_query_id(train_df, val_frac=0.1, seed=42)

train_sub_df = preprocess_data(train_sub_df)
val_df = preprocess_data(val_df)
test_df = preprocess_data(test_df)


train_sub_df = drop_trivial_queries(train_sub_df)
val_df = drop_trivial_queries(val_df)
test_df = drop_trivial_queries(test_df)

In [89]:
from catboost import CatBoostRegressor, CatBoostRanker, CatBoostRanker, Pool


target_col = "relevance"
group_col = "query_id"


feature_cols = [c for c in train_sub_df.columns if c not in [target_col, group_col]]
cat_cols = ["region", "org_id"]


train_sub_df_sort = train_sub_df.sort_values(group_col).reset_index(drop=True)
val_df_sorted = val_df.sort_values(group_col).reset_index(drop=True)
test_df_sorted = test_df.sort_values(group_col).reset_index(drop=True)


train_pool_reg = Pool(
    data=train_sub_df_sort[feature_cols],
    label=train_sub_df_sort[target_col],
    cat_features=cat_cols
)

val_pool_reg = Pool(
    data=val_df_sorted[feature_cols],
    label=val_df_sorted[target_col],
    cat_features=cat_cols
)

test_pool_reg = Pool(
    data=test_df_sorted[feature_cols],
    cat_features=cat_cols
)





train_pool_rank = Pool(
    data=train_sub_df_sort[feature_cols],
    label=train_sub_df_sort[target_col],
    group_id=train_sub_df_sort[group_col],
    cat_features=cat_cols
)

val_pool_rank = Pool(
    data=val_df_sorted[feature_cols],
    label=val_df_sorted[target_col],
    group_id=val_df_sorted[group_col],
    cat_features=cat_cols
)

test_pool_rank = Pool(
    data=test_df_sorted[feature_cols],
    group_id=test_df_sorted[group_col],
    cat_features=cat_cols
)

In [90]:
models = {
    "rmse": CatBoostRegressor(
        loss_function="RMSE",
        eval_metric="RMSE",
        iterations=3000,
        learning_rate=0.05,
        depth=8,
        l2_leaf_reg=6,
        verbose=200,
        random_seed=42,
        od_type="Iter",
        od_wait=100,
    ),
    "pairlogit": CatBoostRanker(
        loss_function="PairLogit",
        eval_metric="NDCG",
        iterations=3000,
        learning_rate=0.05,
        depth=8,
        l2_leaf_reg=6,
        verbose=200,
        random_seed=42,
        od_type="Iter",
        od_wait=100,
    ),
    "yeti_ndcg": CatBoostRanker(
        loss_function="YetiRank:mode=NDCG",
        eval_metric="NDCG",
        iterations=3000,
        learning_rate=0.05,
        depth=8,
        l2_leaf_reg=6,
        verbose=200,
        random_seed=42,
        od_type="Iter",
        od_wait=100,
    ),
}


In [ ]:
results = []
fitted_models = {}
preds_on_test = {}

for name, model in models.items():
    print(f"\n=== Training: {name} ===")

    if isinstance(model, CatBoostRegressor):
        train_pool = train_pool_reg
        val_pool = val_pool_reg
        test_pool = test_pool_reg
    else:
        train_pool = train_pool_rank
        val_pool = val_pool_rank
        test_pool = test_pool_rank

    model.fit(
        train_pool,
        eval_set=val_pool,
        use_best_model=True,
        verbose=200
    )

    val_pred = model.predict(val_pool)
    test_pred = model.predict(test_pool)

    val_scored = val_df_sorted.copy()
    val_scored["pred"] = val_pred

    test_scored = test_df_sorted.copy()
    test_scored["pred"] = test_pred

    val_ndcg10 = compute_mean_ndcg(
        val_scored,
        query_col=group_col,
        target_col=target_col,
        pred_col="pred"
    )

    test_ndcg10 = compute_mean_ndcg(
        test_scored,
        query_col=group_col,
        target_col=target_col,
        pred_col="pred"
    )

    results.append({
        "model": name,
        "val_ndcg": val_ndcg10,
        "test_ndcg": test_ndcg10
    })

    fitted_models[name] = copy.deepcopy(model)
    preds_on_test[name] = test_scored

    print(f"{name} | val NDCG = {val_ndcg10:.5f} | test NDCG = {test_ndcg10:.5f}")


=== Training: rmse ===
0:	learn: 0.0974334	test: 0.0985332	best: 0.0985332 (0)	total: 189ms	remaining: 9m 25s
200:	learn: 0.0851962	test: 0.0923260	best: 0.0923260 (200)	total: 12.7s	remaining: 2m 57s
400:	learn: 0.0827814	test: 0.0918922	best: 0.0917289 (302)	total: 29.5s	remaining: 3m 11s
Stopped by overfitting detector  (100 iterations wait)

bestTest = 0.09172887423
bestIteration = 302

Shrink model to first 303 iterations.
rmse | val NDCG@10 = 0.63248 | test NDCG@10 = 0.60399

=== Training: pairlogit ===
Groupwise loss function. OneHotMaxSize set to 10
0:	test: 0.4975723	best: 0.4975723 (0)	total: 100ms	remaining: 5m
Stopped by overfitting detector  (100 iterations wait)

bestTest = 0.4981433467
bestIteration = 12

Shrink model to first 13 iterations.
pairlogit | val NDCG@10 = 0.62862 | test NDCG@10 = 0.59901

=== Training: yeti_ndcg ===
Groupwise loss function. OneHotMaxSize set to 10
0:	test: 0.4975723	best: 0.4975723 (0)	total: 78.1ms	remaining: 3m 54s
Stopped by overfitting d

In [95]:
pd.DataFrame(results).sort_values("val_ndcg", ascending=False)

,model,val_ndcg,test_ndcg
0,rmse,0.632481,0.603993
2,yeti_ndcg,0.629448,0.598223
1,pairlogit,0.628621,0.599013


**Задание 17. (1 балл).** Одним из основных преимуществ CatBoost'a является обработка категориальных факторов «из коробки». Добавьте в датасет различные категориальные факторы из данных и обучите заново CatBoost модели. Улучшилось ли качество?

**Задание 18. (3 бонусных балла)**

Для тех кто дожил до самого конца последней большой лабы есть бонус в виде рисеч задания. Здесь вам предстоит сделать обзор на 2 статьи из современного рексиса. В качестве основы для поиска статей и введение в курс дела предлагается прочитать статью с хабра))): https://habr.com/ru/companies/yandex/articles/857068/

Выберите 2 тренда, которые вам больше всего понравились и сделайте обзор ниже на каждую из статью внутри этого тренда. Во время обзора постарайтесь ограничиться парочкой абзацев и побольше картинок - проверяющему главное увидеть, что вы поняли главную идею статьи.